#1. IMPORTS

In [8]:
import os
import re
import sys
import unicodedata

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml


def encontrar_project_root():
    actual = Path.cwd().resolve()

    for ruta in [actual, *actual.parents]:
        if (
            (ruta / "config" / "config.yaml").exists()
            and
            (ruta / "src").exists()
        ):
            return ruta

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto."
    )


PROJECT_ROOT = encontrar_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from src.config import load_config, get_data_root


config = load_config()
DATA_ROOT = get_data_root(config)

TRAIN_PATH = (
    DATA_ROOT
    / config["data"]["train_csv"]
)

WEAK_DATA_DIR = (
    DATA_ROOT
    / config["data"]["processed"]["weak_labels"]
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "weak_labels"
)

MANIFEST_DIR = (
    DATA_ROOT
    / config["data"]["manifests"]["directory"]
)

WEAK_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_ROOT    :", DATA_ROOT)
print("TRAIN_PATH   :", TRAIN_PATH)
print("Existe train:", TRAIN_PATH.exists())


gpu_available = torch.cuda.is_available()

print(
    "\n¿Hay GPU disponible?:",
    gpu_available
)

if gpu_available:
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )
else:
    print(
        "Dispositivo activo: CPU"
    )

PROJECT_ROOT : C:\Users\PC\Documents\knee-mri-weak-supervision
DATA_ROOT    : C:\Users\PC\Documents\RSNA_Knee
TRAIN_PATH   : C:\Users\PC\Documents\RSNA_Knee\raw\train.csv
Existe train: True

¿Hay GPU disponible?: False
Dispositivo activo: CPU


#2. CARGA DE DATOS y CREAR GOLD/weak

In [9]:
if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró train.csv en:\n{TRAIN_PATH}"
    )


df = pd.read_csv(
    TRAIN_PATH
)


print(
    "Dimensiones del DataFrame completo:",
    df.shape
)


labels = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]


gold_mask = (
    df[labels]
    .notna()
    .all(axis=1)
)

weak_mask = (
    df[labels]
    .isna()
    .all(axis=1)
)

partial_mask = ~(
    gold_mask
    |
    weak_mask
)


gold_df = df[
    gold_mask
].copy()

weak_df = df[
    weak_mask
].copy()


print("Total estudios:", len(df))
print("Estudios GOLD:", len(gold_df))
print("Estudios weak:", len(weak_df))
print(
    "Estudios parcialmente etiquetados:",
    partial_mask.sum()
)


assert df.shape == (4407, 14)
assert len(gold_df) == 58
assert len(weak_df) == 4349
assert partial_mask.sum() == 0


print(
    "\n✓ Dataset validado correctamente"
)

Dimensiones del DataFrame completo: (4407, 14)
Total estudios: 4407
Estudios GOLD: 58
Estudios weak: 4349
Estudios parcialmente etiquetados: 0

✓ Dataset validado correctamente


#3. Detectar Idioma

In [10]:
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 42


def detectar_idioma(texto):
    try:
        return detect(
            str(texto)
        )
    except Exception:
        return "unknown"


report_col = (
    'Report"'
    if 'Report"' in df.columns
    else "Report"
)


df["Idioma"] = (
    df[report_col]
    .apply(detectar_idioma)
)

gold_df["Idioma"] = (
    df.loc[
        gold_df.index,
        "Idioma"
    ]
)

weak_df["Idioma"] = (
    df.loc[
        weak_df.index,
        "Idioma"
    ]
)


print(
    "Distribución de idiomas:"
)

print(
    df["Idioma"]
    .value_counts()
)

Distribución de idiomas:
Idioma
en    1736
es     682
tr     546
hr     406
el     321
de     262
bg     220
nl     153
fr      81
Name: count, dtype: int64


#4. Split

In [11]:
from iterstrat.ml_stratifiers import (
    MultilabelStratifiedShuffleSplit
)


X = np.arange(
    len(gold_df)
).reshape(-1, 1)

y = (
    gold_df[labels]
    .astype(int)
    .values
)


splitter = (
    MultilabelStratifiedShuffleSplit(
        n_splits=1,
        test_size=0.30,
        random_state=42
    )
)


idx_dev, idx_holdout = next(
    splitter.split(
        X,
        y
    )
)


gold_dev = (
    gold_df
    .iloc[idx_dev]
    .copy()
)

gold_holdout = (
    gold_df
    .iloc[idx_holdout]
    .copy()
)


print(
    "GOLD DEV:",
    len(gold_dev)
)

print(
    "GOLD HOLDOUT:",
    len(gold_holdout)
)


assert len(gold_dev) == 40
assert len(gold_holdout) == 18

GOLD DEV: 40
GOLD HOLDOUT: 18


In [12]:
gold_split = pd.concat(
    [
        gold_dev[
            ["StudyInstanceUID"]
        ].assign(split="DEV"),

        gold_holdout[
            ["StudyInstanceUID"]
        ].assign(split="HOLDOUT")
    ],
    ignore_index=True
)


GOLD_SPLIT_PATH = (
    MANIFEST_DIR
    / "gold_split.csv"
)


gold_split.to_csv(
    GOLD_SPLIT_PATH,
    index=False
)


print(
    "✓ Split GOLD guardado en:",
    GOLD_SPLIT_PATH
)

✓ Split GOLD guardado en: C:\Users\PC\Documents\RSNA_Knee\manifest\gold_split.csv


#5. FUNCIONES AUXILIARES

##5.1. Normalizar Texto

In [13]:
def normalizar_texto(texto):
    texto = str(texto).lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))

    # Manejo de caracteres especiales en otros idiomas
    texto = texto.replace("ı", "i")
    texto = texto.replace("ß", "ss")

    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()

##5.2. Dividir en Fragmentos

In [14]:
def dividir_en_fragmentos(reporte):
    texto = str(reporte)
    # Convertir saltos de línea en espacios para no romper frases médicas
    texto = re.sub(r"\s*\n\s*", " ", texto)
    # Dividir por puntuación real
    partes = re.split(r"[.;]+", texto)
    return [
        normalizar_texto(parte)
        for parte in partes
        if parte.strip()
    ]

##5.3. Contiene()

In [15]:
def contiene(texto, patrones):
    return any(re.search(patron, texto) for patron in patrones)

##5.4. Función Evaluar Extractor

In [16]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluar_extractor(df,etiqueta_real,columna_pred,conjunto="DEV"):
    resueltos = df[df[columna_pred] != -1].copy()
    aciertos = (resueltos[etiqueta_real].astype(int)== resueltos[columna_pred].astype(int)).sum()
    total = len(resueltos)
    accuracy = aciertos / total if total > 0 else 0
    coverage = total / len(df)

    y_true = resueltos[etiqueta_real].astype(int)
    y_pred = resueltos[columna_pred].astype(int)

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    print(f"Casos evaluados ({conjunto}): {len(df)}")
    print(
        f"Casos resueltos "
        f"(Cobertura {coverage*100:.1f}%): {total}"
    )
    print(
        f"Aciertos: {aciertos} | "
        f"Errores: {total - aciertos}"
    )
    print(f"Accuracy: {round(accuracy, 3)}")
    print(
        f"Precision: {round(precision, 3)} | "
        f"Recall: {round(recall, 3)} | "
        f"F1: {round(f1, 3)}"
    )


#6. Extractores de etiquetas débiles

Cada extractor devuelve:

- `1`: hallazgo presente
- `0`: hallazgo ausente
- `-1`: evidencia insuficiente / abstención

Los extractores se desarrollaron utilizando únicamente `gold_dev`
y posteriormente se congelaron antes de evaluar `gold_holdout`.

# **ACL**

In [17]:
# @title
def extraer_acl(reporte):

    fragmentos = dividir_en_fragmentos(reporte)

    evidencia_positiva = False
    evidencia_negativa = False
    patrones_acl = [
        # Inglés
        r"\bacl\b",
        r"anterior cruciate ligament",

        # Español
        r"\blca\b",
        r"ligamento cruzado anterior",

        # Turco
        r"on capraz bag",
        r"anterior capraz bag",

        # Alemán
        r"\bvkb\b",
        r"vorder.{0,10}kreuzband",

        # Neerlandés
        r"voorste kruisband",

        # Croata
        r"prednji krizni ligament",

        # Búlgaro
        r"предн.{0,25}кръстн.{0,25}връз",

        # Griego
        r"προσθι.{0,30}χιαστ"
    ]
    patrones_fuertemente_positivos = [
        r"complete tear",
        r"complete rupture",
        r"high[- ]grade tear",
        r"high[- ]grade partial tear",
        r"full[- ]thickness",
        r"rotura completa",
        r"ruptura completa",
        r"rotura espesor y ancho total",
        r"\bruptur",
        r"\briss\b",
        r"\byirtik\b",
        r"\bkomplet\b",
        r"potpuna ruptura",
        r"скъсване",
        r"руптура",
        r"ρηξ",
        r"ασυνεχεια"
    ]
    patrones_bajo_grado = [
        r"low[- ]grade",
        r"interstitial tear",
        r"grade\s*1\b",
        r"grade\s*2\b",
        r"grade\s*i\b",
        r"grade\s*ii\b",
        r"\bsprain\b",
        r"minor partial",
        r"small partial",
        r"manje parcijalne rupture"
    ]
    patrones_normales = [
        r"\bintact\b",
        r"\bnormal\b",
        r"\bpreserved\b",
        r"\bunremarkable\b",
        r"within normal",
        r"sin alteraciones",
        r"conservado",
        r"dentro de limites normales",
        r"\bintakt\b",
        r"\bnormaal\b",
        r"\bnormaldir\b",
        r"\bkorunmus\b",
        r"\buredan\b",
        r"\bocuvan\b",
        r"запазена",
        r"запазени",
        r"φυσιολογ"
    ]
    for fragmento in fragmentos:

        # Solo analizamos frases que realmente hablan del ACL
        habla_acl = contiene(
            fragmento,
            patrones_acl
        )

        if not habla_acl:
            continue

        # 1. Lesión fuerte
        fuerte = contiene(
            fragmento,
            patrones_fuertemente_positivos
        )

        # 2. Lesión leve
        leve = contiene(
            fragmento,
            patrones_bajo_grado
        )

        # 3. ACL descrito como normal
        normal = contiene(
            fragmento,
            patrones_normales
        )

        if fuerte:
            evidencia_positiva = True

        elif leve:
            evidencia_negativa = True

        elif normal:
            evidencia_negativa = True


    # Casos especiales donde dicen que ambos ligamentos
    # cruzados son normales sin mencionar ACL/LCA explícitamente

    texto = normalizar_texto(reporte)

    cruzados_normales = [
        r"ligamentos cruzados.{0,80}"
        r"(normales|dentro de limites normales|sin alteraciones)",

        r"(normales|dentro de limites normales|sin alteraciones)"
        r".{0,80}ligamentos cruzados",

        # Griego: "normalmente representados los ligamentos cruzados"
        r"φυσιολογ.{0,100}χιαστ",

        # Búlgaro
        r"кръстн.{0,50}връзк.{0,80}"
        r"(запазена|запазени|нормал)"
    ]

    if contiene(texto, cruzados_normales):
        evidencia_negativa = True


    # Una ruptura fuerte gana frente a menciones
    # de normalidad de otras estructuras
    if evidencia_positiva:
        return 1

    if evidencia_negativa:
        return 0

    return -1

In [18]:
# Aplicar extractor ACL sobre gold_dev
gold_dev["ACL_pred"] = (
    gold_dev["Report"]
    .apply(extraer_acl)
)

print("=== EVALUACIÓN ACL ===")

evaluar_extractor(
    gold_dev,
    "ACL",
    "ACL_pred"
)

=== EVALUACIÓN ACL ===
Casos evaluados (DEV): 40
Casos resueltos (Cobertura 70.0%): 28
Aciertos: 27 | Errores: 1
Accuracy: 0.964
Precision: 0.929 | Recall: 1.0 | F1: 0.963


# **Medial Meniscus**

In [19]:
# @title
def extraer_medial_meniscus(reporte):

    fragmentos = dividir_en_fragmentos(reporte)

    patrones_medial = [

        # Inglés
        r"medial meniscus",

        # Español
        r"menisco medial",
        r"menisco interno",

        # Turco
        r"medial menisk",

        # Croata
        r"medijalni menisk",

        # Alemán
        r"innenmenisk",

        # Neerlandés
        r"mediale meniscus",

        # Francés
        r"menisque medial",

        # Búlgaro
        r"медиалн.{0,20}мениск",

        # Griego
        r"εσω.{0,25}μηνισκ"
    ]


    positivos_fuertes = [

        # Inglés
        r"\btear\b",
        r"\btorn\b",
        r"\brupture\b",
        r"bucket.?handle",
        r"radial tear",
        r"root tear",
        r"complex tear",

        # Español
        r"\brotura\b",
        r"\bruptura\b",
        r"\bdesgarro\b",

        # Turco
        r"\byirtik\b",

        # Croata
        r"\bruptur",

        # Alemán
        r"\briss\b",
        r"radiale lasion",
        r"separation.{0,40}lappen",

        # Neerlandés
        r"\bscheur\b",

        # Búlgaro
        r"руптура",
        r"скъсване",

        # Griego
        r"ρηξ"
    ]


    negativos_claros = [

        # Inglés
        r"not torn",
        r"\bno tear\b",
        r"no frank tear",
        r"\bnormal\b",
        r"\bintact\b",
        r"\bpreserved\b",

        # Español
        r"sin signos de rotura",
        r"sin signos de desgarro",
        r"sin evidencia de rotura",
        r"sin evidencia de desgarro",
        r"sin criterios categoricos de rotura",
        r"sin caracteres propios de rotura",
        r"morfologia y señal conservada",
        r"morfologia conservada",

        # Turco
        r"\bnormal\b",

        # Croata
        r"bez znakova rupture",

        # Alemán
        r"kein.{0,20}riss",
        r"\bintakt\b",

        # Neerlandés
        r"normaal voorkomen",

        # Búlgaro
        r"нормално представени",

        # Griego
        r"φυσιολογ"
    ]


    ambiguos = [

        r"\bsuspect",
        r"\bsuspicious",
        r"\br/o\b",
        r"rule out",
        r"\bpossible\b",
        r"could represent",
        r"may represent",

        r"\bsugier",
        r"\bpodria\b",
        r"\bposible\b",

        # Cambios degenerativos sin rotura confirmada
        r"intrasubstance degeneration",
        r"mucoid degeneration"
    ]


    evidencia_positiva = False
    evidencia_negativa = False
    evidencia_ambigua = False


    for fragmento in fragmentos:

        habla_medial = contiene(
            fragmento,
            patrones_medial
        )

        # Caso especial griego:
        # "menisco en el compartimento medial"
        if (
            not habla_medial
            and re.search(r"εσω.{0,80}διαμερισ", fragmento)
            and re.search(r"μηνισκ", fragmento)
        ):
            habla_medial = True

        if not habla_medial:
            continue

        ambiguo = contiene(
            fragmento,
            ambiguos
        )

        negativo = contiene(
            fragmento,
            negativos_claros
        )

        positivo = contiene(
            fragmento,
            positivos_fuertes
        )

        # Una negación explícita como
        # "sin signos de desgarro"
        # tiene prioridad
        if negativo:
            evidencia_negativa = True

        elif ambiguo:
            evidencia_ambigua = True

        elif positivo:
            evidencia_positiva = True


    # Si encontramos afirmaciones contradictorias,
    # preferimos no inventar.
    if evidencia_positiva and evidencia_negativa:
        return -1

    if evidencia_positiva:
        return 1

    if evidencia_negativa:
        return 0

    return -1

In [20]:
# 1. Aplicar el extractor de Menisco Medial a gold_dev
label_actual = "Medial Meniscus"
columna_pred = "Medial_Meniscus_pred"

gold_dev[columna_pred] = gold_dev["Report"].apply(extraer_medial_meniscus)

print(f"=== EVALUACIÓN {label_actual.upper()} ===")
evaluar_extractor(gold_dev, label_actual, columna_pred)

# MEDIAL MENISCUS
gold_dev["Medial_Meniscus_pred"] = (
    gold_dev["Report"]
    .apply(extraer_medial_meniscus)
)

=== EVALUACIÓN MEDIAL MENISCUS ===
Casos evaluados (DEV): 40
Casos resueltos (Cobertura 67.5%): 27
Aciertos: 25 | Errores: 2
Accuracy: 0.926
Precision: 0.895 | Recall: 1.0 | F1: 0.944


# **MCL**

In [21]:
# @title
# --- 2. EXTRACTOR MCL V2 ---

def extraer_MCL(reporte):

    fragmentos = dividir_en_fragmentos(reporte)
    texto = normalizar_texto(reporte)

    patrones_mcl = [
        # Inglés
        r"\bmcl\b",
        r"medial collateral ligament",

        # Español
        r"\blcm\b",
        r"ligamento colateral medial",

        # Neerlandés
        r"mediale collaterale ligament",

        # Turco
        r"medial kollateral ligam",

        # Croata
        r"medijalni kolateralni ligament",

        # Alemán
        r"mediales seitenband"
    ]

    # Evidencia positiva muy fuerte
    positivos_fuertes = [
        r"complete tear",
        r"complete rupture",
        r"high[- ]grade tear",
        r"high[- ]grade injury",
        r"ruptura completa",
        r"rotura completa",
        r"perdida completa de la continuidad"
    ]

    # Evidencia negativa clara
    negativos_claros = [
        r"\bintact\b",
        r"\bnormal\b",
        r"\bpreserved\b",
        r"\bunremarkable\b",
        r"within normal",
        r"sin alteraciones",
        r"dentro de limites normales",
        r"conservado",
        r"low[- ]grade",
        r"grade\s*1\b",
        r"grade\s*i\b",
        r"no frank tear"
    ]

    # Casos ambiguos
    ambiguos = [
        r"grade\s*2\b",
        r"grade\s*ii\b",
        r"partial tear",
        r"esguince grado\s*ii",
        r"graad\s*2",
        r"partiele scheur"
    ]

    # Expresiones generales que también implican MCL normal
    patrones_grupales_negativos = [

        # Español
        r"ligamentos cruzados y colaterales.{0,70}"
        r"(dentro de limites normales|sin alteraciones)",

        r"ligamentos colaterales.{0,70}"
        r"(dentro de limites normales|sin alteraciones)",

        # Inglés
        r"medial and lateral collateral ligament"
        r".{0,60}(intact|normal|preserved)",

        r"collateral ligament complexes.{0,60}"
        r"(intact|normal|preserved)",

        # Alemán
        r"\bmcl\b.{0,30}\blcl\b.{0,50}intakt",

        # Griego
        r"φυσιολογ.{0,100}πλαγι.{0,50}συνδεσμ",

        # Búlgaro
        r"колатералн.{0,80}"
        r"(правилна|нормал|запазен)"
    ]

    evidencia_positiva = False
    evidencia_negativa = False
    evidencia_ambigua = False

    # 1. Primero evaluamos las expresiones grupales
    #    sobre TODO el reporte
    if contiene(texto, patrones_grupales_negativos):
        evidencia_negativa = True

    # 2. Luego analizamos fragmentos específicos del MCL
    for fragmento in fragmentos:

      if not contiene(fragmento, patrones_mcl):
          continue

      if contiene(fragmento, positivos_fuertes):
          evidencia_positiva = True

      elif contiene(fragmento, negativos_claros):
          evidencia_negativa = True

      elif contiene(fragmento, ambiguos):
          evidencia_ambigua = True

    # 3. Decisión final

    # Si existen afirmaciones fuertes contradictorias:
    if evidencia_positiva and evidencia_negativa:
        return -1

    # Lesión inequívocamente fuerte
    if evidencia_positiva:
        return 1

    # Grade II / partial tear ambiguo
    if evidencia_ambigua:
        return -1

    # Normalidad clara
    if evidencia_negativa:
        return 0

    return -1

In [22]:
label_actual = "MCL"
columna_pred = "MCL_pred"

gold_dev[columna_pred] = (
    gold_dev["Report"]
    .apply(extraer_MCL)
)

print(
    f"=== EVALUACIÓN {label_actual.upper()} ==="
)

evaluar_extractor(
    gold_dev,
    label_actual,
    columna_pred
)

=== EVALUACIÓN MCL ===
Casos evaluados (DEV): 40
Casos resueltos (Cobertura 60.0%): 24
Aciertos: 24 | Errores: 0
Accuracy: 1.0
Precision: 1.0 | Recall: 1.0 | F1: 1.0


# **Lateral Meniscus**

In [23]:
# @title
def extraer_lateral_meniscus(reporte):

    fragmentos = dividir_en_fragmentos(reporte)

    patrones_lateral = [

        # Inglés
        r"lateral meniscus",
        r"lateral meniscal",

        # Español
        r"menisco lateral",

        # Turco
        r"lateral menisk",

        # Croata
        r"lateralni menisk",

        # Alemán
        r"aussenmenisk",

        # Neerlandés
        r"laterale meniscus",

        # Francés
        r"menisque lateral",

        # Búlgaro
        r"латералн.{0,20}мениск",

        # Griego
        r"εξω.{0,25}μηνισκ"
    ]


    positivos_fuertes = [

        # Inglés
        r"\btear\b",
        r"\btearing\b",          # NUEVO
        r"\btorn\b",
        r"\brupture\b",
        r"bucket.?handle",
        r"radial tear",
        r"root tear",
        r"complex tear",
        r"longitudinal tear",

        # Español
        r"\brotura\b",
        r"\bruptura\b",
        r"\bdesgarro\b",

        # Turco:
        # yırtık / yırtığı / yırtıklar...
        r"\byirti(?:k|g)\w*",    # NUEVO

        # Croata
        r"\bruptur",

        # Alemán
        r"\briss\b",

        # Neerlandés
        r"\bscheur\b",

        # Búlgaro
        r"руптура",
        r"скъсване",

        # Griego
        r"ρηξ"
    ]


    negativos_claros = [

        # Inglés
        r"no lateral meniscal tear",       # NUEVO
        r"no lateral meniscus tear",       # NUEVO
        r"lateral meniscus.{0,30}no tear", # NUEVO
        r"lateral meniscus.{0,30}intact",  # NUEVO
        r"lateral meniscus.{0,30}normal",  # NUEVO
        r"not torn",
        r"\bno tear\b",
        r"no obvious tear",
        r"no frank tear",
        r"\bnormal\b",
        r"\bintact\b",
        r"\bpreserved\b",

        # Español
        r"sin signos de rotura",
        r"sin signos de desgarro",
        r"sin evidencia de rotura",
        r"sin evidencia de desgarro",
        r"morfologia y señal conservada",
        r"morfologia conservada",

        # Turco
        r"lateral menisk.{0,30}normal",

        # Croata
        r"bez znakova rupture",

        # Neerlandés
        r"normaal voorkomen",

        # Búlgaro
        r"нормално представени"
    ]


    ambiguos = [

        r"\bsuspect",
        r"\bsuspicious",
        r"\br/o\b",
        r"rule out",
        r"\bpossible\b",
        r"may represent",
        r"could represent",

        r"\bposible\b",
        r"\bpodria\b",
        r"\bsugier",

        # Degeneración aislada no implica necesariamente tear
        r"grade\s*1 degeneration",
        r"grade\s*i degeneration",
        r"grade\s*2 degeneration",
        r"grade\s*ii degeneration",
        r"degenerative signal"
    ]


    evidencia_positiva = False
    evidencia_negativa = False
    evidencia_ambigua = False


    for fragmento in fragmentos:

        if not contiene(fragmento, patrones_lateral):
            continue

        negativo = contiene(
            fragmento,
            negativos_claros
        )

        ambiguo = contiene(
            fragmento,
            ambiguos
        )

        positivo = contiene(
            fragmento,
            positivos_fuertes
        )

        # Negación explícita tiene prioridad
        if negativo:
            evidencia_negativa = True

        elif ambiguo:
            evidencia_ambigua = True

        elif positivo:
            evidencia_positiva = True


    if evidencia_positiva and evidencia_negativa:
        return -1

    if evidencia_positiva:
        return 1

    if evidencia_negativa:
        return 0

    return -1

In [24]:
# --- APLICACIÓN Y EVALUACIÓN LATERAL MENISCUS ---

label_actual = "Lateral Meniscus"
columna_pred = "Lateral_Meniscus_pred"

gold_dev[columna_pred] = (
    gold_dev["Report"]
    .apply(extraer_lateral_meniscus)
)

print(
    f"=== EVALUACIÓN {label_actual.upper()} ==="
)

evaluar_extractor(
    gold_dev,
    label_actual,
    columna_pred
)

=== EVALUACIÓN LATERAL MENISCUS ===
Casos evaluados (DEV): 40
Casos resueltos (Cobertura 60.0%): 24
Aciertos: 22 | Errores: 2
Accuracy: 0.917
Precision: 0.923 | Recall: 0.923 | F1: 0.923


# **Medial OA**

In [25]:
# @title
# --- EXTRACTOR MEDIAL OA V4 STANDALONE ---

def extraer_Medial_OA(reporte):

    texto = normalizar_texto(reporte)

    evidencia_positiva = False
    evidencia_negativa = False


    # ==================================================
    # 1. POSITIVOS FUERTES
    # ==================================================

    positivos_fuertes = [

        # ----------------------------------------------
        # Inglés
        # ----------------------------------------------

        r"oa of medial compartment",

        r"medial compartment.{0,120}"
        r"(osteoarthritis|osteoarthrosis)",

        r"(osteoarthritis|osteoarthrosis)"
        r".{0,120}medial compartment",

        r"medial tibiofemoral.{0,120}"
        r"(osteoarthritis|osteoarthrosis|\boa\b)",

        r"medial femorotibial.{0,120}"
        r"(osteoarthritis|osteoarthrosis|\boa\b)",

        # Inglés: daño condral medial fuerte
        r"medial compartment cartilage.{0,120}"
        r"high[- ]grade cartilage fissur",

        r"high[- ]grade cartilage fissur.{0,120}"
        r"medial femoral condyle",


        # ----------------------------------------------
        # Español
        # ----------------------------------------------

        r"femorotibial medial.{0,120}"
        r"(osteoartrosis|artrosis|\boa\b)",

        r"(osteoartrosis|artrosis|\boa\b)"
        r".{0,120}femorotibial medial",

        r"compartimento femorotibial medial.{0,120}"
        r"(osteoartrosis|artrosis|\boa\b)",

        r"(osteoartrosis|artrosis)"
        r".{0,120}compartimento medial",
        # Español: lesión grado 3/4 en región medial
        r"condropatia.{0,120}"
        r"grado\s*(?:3|4|iii|iv)"
        r".{0,160}"
        r"condilo femoral medial",

        r"condropatia.{0,120}"
        r"grado\s*(?:3|4|iii|iv)"
        r".{0,160}"
        r"compartimento femorotibial medial",

        # Úlceras condrales fuertes en compartimento medial
        r"ulceras condrales.{0,180}"
        r"(espesor parcial y total|espesor total)"
        r".{0,180}"
        r"condilo femoral medial",


        # ----------------------------------------------
        # OA TRICOMPARTIMENTAL
        # incluye compartimento medial
        # ----------------------------------------------

        r"tricompartmental.{0,80}"
        r"(osteoarthritis|osteoarthrosis)",

        r"(osteoarthritis|osteoarthrosis)"
        r".{0,80}tricompartmental",

        r"oa of all three compart\w*",

        r"incipient oa of all three compart\w*",


        # ----------------------------------------------
        # Turco
        # ----------------------------------------------

        r"medyal femorotibyal.{0,80}"
        r"(osteoartrit|osteoartroz)",

        r"medial femorotibyal.{0,80}"
        r"(osteoartrit|osteoartroz)",

        r"medial kompart.{0,100}"
        r"(osteoartrit|osteoartroz)",


        # ----------------------------------------------
        # Griego
        # ----------------------------------------------

        r"οστεοαρθρ.{0,150}"
        r"εσω.{0,60}διαμερισ",

        r"εσω.{0,60}διαμερισ.{0,150}"
        r"οστεοαρθρ",


        # ----------------------------------------------
        # Alemán
        # ----------------------------------------------

        r"medial.{0,30}kompartiment.{0,120}"
        r"(arthrose|osteoarth)",


        # ----------------------------------------------
        # Neerlandés
        # ----------------------------------------------

        r"mediaal femorotibiaal.{0,120}"
        r"(artrose|osteoartrose)",

        r"mediale compartiment.{0,120}"
        r"(artrose|osteoartrose)",


        # ----------------------------------------------
        # Croata
        # ----------------------------------------------

        r"medijaln.{0,30}kompartment.{0,120}"
        r"(osteoartroz|\boa\b)"
    ]


    # ==================================================
    # 2. NEGATIVOS BASE
    # Ya estaban en la lógica anterior
    # ==================================================

    negativos_base = [

        # Inglés
        r"medial compartment cartilage.{0,60}"
        r"(intact|normal|preserved)",

        r"medial compartment.{0,100}"
        r"no focal chondrosis",

        r"medial compartment.{0,100}"
        r"no chondral injury",

        r"preserved medial.{0,60}"
        r"(cartilage|articular cartilage)",

        r"medial femorotibial.{0,80}"
        r"(cartilage is normal|cartilage normal|preserved)",


        # Español
        r"compartimento femorotibial medial.{0,100}"
        r"sin alteraciones",

        r"cartilago.{0,80}"
        r"femorotibial medial.{0,80}"
        r"sin alteraciones"
    ]


    # ==================================================
    # 3. NEGATIVOS AÑADIDOS EN V4
    # ==================================================

    negativos_v4 = [

        # -------------------------
        # Español
        # -------------------------

        r"cartilagos de los compartimentos femorotibiales"
        r".{0,60}sin alteraciones",

        r"resto de los cartilagos femorotibiales"
        r".{0,80}sin alteraciones",

        r"cartilagos femorotibiales"
        r".{0,80}sin alteraciones",


        # -------------------------
        # Inglés
        # -------------------------

        r"medial compartment.{0,220}"
        r"no focal chondrosis or chondral injury",

        r"medial compartment.{0,220}"
        r"no focal chondrosis",

        r"preserved medial and lateral femorotibial"
        r".{0,80}articular cartilage",

        r"tibiofemoral and patellofemoral"
        r".{0,80}articular cartilage.{0,30}normal",

        r"medial and lateral femorotibial"
        r".{0,80}cartilage.{0,30}(normal|preserved)",

        # Inglés
        r"\bcartilages?\s+normal\b",

        # Neerlandés
        r"normaal voorkomen kraakbeen",


        # -------------------------
        # Turco
        # -------------------------

        r"eklem k[iı]k[iı]rdaklar[iı].{0,80}"
        r"(normal|korunmus)",


        # -------------------------
        # Búlgaro
        # -------------------------

        r"хрущял.{0,120}"
        r"(съхранен|интактен)"
    ]


    # ==================================================
    # 4. EVIDENCIA
    # ==================================================

    if contiene(texto, positivos_fuertes):
        evidencia_positiva = True

    if contiene(texto, negativos_base):
        evidencia_negativa = True


    # ==================================================
    # 5. DECISIÓN EQUIVALENTE A V3
    # ==================================================

    # Si el texto tiene evidencia contradictoria,
    # preferimos abstenernos
    if evidencia_positiva and evidencia_negativa:
        return -1

    # OA medial explícita
    if evidencia_positiva:
        return 1

    # Normalidad medial clara
    if evidencia_negativa:
        return 0

    if contiene(texto, negativos_v4):
        return 0


    # ==================================================
    # 7. SIN EVIDENCIA SUFICIENTE
    # ==================================================

    return -1

In [26]:
columna_pred = "Medial_OA_pred"

gold_dev[columna_pred] = (
    gold_dev["Report"]
    .apply(extraer_Medial_OA)
)

evaluar_extractor(
    gold_dev,
    "Medial OA",
    columna_pred
)

print(
    gold_dev[columna_pred]
    .value_counts()
)

Casos evaluados (DEV): 40
Casos resueltos (Cobertura 72.5%): 29
Aciertos: 27 | Errores: 2
Accuracy: 0.931
Precision: 0.818 | Recall: 1.0 | F1: 0.9
Medial_OA_pred
 0    18
-1    11
 1    11
Name: count, dtype: int64


#Lateral OA

In [27]:
# @title
# --- EXTRACTOR LATERAL OA V5 ---

def extraer_Lateral_OA(reporte):

    texto = normalizar_texto(reporte)

    evidencia_positiva = False
    evidencia_negativa = False
    evidencia_ambigua = False


    # ==================================================
    # 1. POSITIVOS CLAROS
    # ==================================================

    positivos_claros = [

        # ----------------------------------------------
        # OA lateral explícita - Inglés
        # ----------------------------------------------

        r"oa of (?:the )?lateral compartment",

        r"lateral compartment.{0,120}"
        r"(osteoarthritis|osteoarthrosis)",

        r"(osteoarthritis|osteoarthrosis)"
        r".{0,120}lateral compartment",

        r"lateral tibiofemoral.{0,120}"
        r"(osteoarthritis|osteoarthrosis|\boa\b)",


        # ----------------------------------------------
        # Español
        # ----------------------------------------------

        r"femorotibial lateral.{0,120}"
        r"(osteoartrosis|artrosis|\boa\b)",

        r"(osteoartrosis|artrosis|\boa\b)"
        r".{0,120}femorotibial lateral",


        # Condropatía grado 3 / 4 del compartimento lateral
        r"condropatia.{0,100}"
        r"grado\s*(?:3|4|iii|iv)"
        r".{0,180}"
        r"compartimento femorotibial lateral",

        r"compartimento femorotibial lateral.{0,180}"
        r"condropatia.{0,100}"
        r"grado\s*(?:3|4|iii|iv)",


        # Daño significativo en estructuras laterales
        r"condropatia.{0,160}"
        r"grado\s*(?:3|4|iii|iv)"
        r".{0,200}"
        r"(platillo tibial lateral|condilo femoral lateral)",

        r"(platillo tibial lateral|condilo femoral lateral)"
        r".{0,200}"
        r"condropatia.{0,120}"
        r"grado\s*(?:3|4|iii|iv)",


        # Úlcera condral importante lateral
        r"ulceras? condrales?.{0,180}"
        r"(espesor total|grado\s*(?:3|4|iii|iv))"
        r".{0,200}"
        r"(platillo tibial lateral|condilo femoral lateral)",


        # ----------------------------------------------
        # Inglés: daño degenerativo lateral extenso
        # ----------------------------------------------

        r"lateral compartment.{0,500}"
        r"full[- ]thickness fissur.{0,300}"
        r"moderate thinning",

        r"lateral compartment.{0,350}"
        r"diffuse.{0,120}"
        r"(high[- ]grade|full[- ]thickness)"
        r".{0,150}"
        r"(chondrosis|cartilage loss)",


        # ----------------------------------------------
        # OA en los tres compartimentos
        # ----------------------------------------------

        r"incipient oa of all three compart\w*",

        r"oa of all three compart\w*",

        r"tricompartmental.{0,60}"
        r"(osteoarthritis|osteoarthrosis|\boa\b)",


        # ----------------------------------------------
        # Neerlandés / Croata
        # ----------------------------------------------

        r"lateraal femorotibiaal.{0,180}"
        r"(kraakbeenverlies|kraakbeenlijden)",

        r"lateraln.{0,50}kompartment.{0,200}"
        r"(hrskavic|erozij|fisur)"
    ]


    # ==================================================
    # 2. NEGATIVOS CLAROS
    # ==================================================

    negativos_claros = [

        # ----------------------------------------------
        # Inglés: lateral explícitamente normal
        # ----------------------------------------------

        r"lateral compartment.{0,200}"
        r"no focal chondrosis or chondral injury",

        r"lateral compartment.{0,200}"
        r"no focal chondrosis",

        r"lateral compartment cartilage.{0,100}"
        r"(intact|normal|preserved)",

        r"lateral femorotibial cartilage.{0,100}"
        r"(intact|normal|preserved)",


        # Ambos compartimentos tibiofemorales normales
        r"preserved medial and lateral femorotibial"
        r".{0,80}articular cartilage",

        r"medial and lateral femorotibial"
        r".{0,120}articular cartilage"
        r".{0,50}(normal|preserved|intact)",

        r"tibiofemoral and patellofemoral"
        r".{0,120}articular cartilage"
        r".{0,50}(normal|preserved|intact)",


        # ----------------------------------------------
        # OA explícitamente MEDIAL
        # sin evidencia positiva lateral
        # ----------------------------------------------

        r"\boa of (?:the )?medial compartment\b",

        r"medial compartment.{0,100}"
        r"(osteoarthritis|osteoarthrosis)",

        r"(osteoarthritis|osteoarthrosis)"
        r".{0,100}medial compartment",


        # ----------------------------------------------
        # Español
        # ----------------------------------------------

        r"cartilagos de los compartimentos femorotibiales"
        r".{0,120}sin alteraciones",

        r"resto de los cartilagos femorotibiales"
        r".{0,120}sin alteraciones",

        r"compartimento femorotibial lateral"
        r".{0,120}sin alteraciones",

        # OA solo medial
        r"(osteoartrosis|artrosis).{0,100}"
        r"compartimento medial",

        r"compartimento medial.{0,100}"
        r"(osteoartrosis|artrosis)",


        # ----------------------------------------------
        # Griego:
        # OA localizada en compartimento medial
        # ----------------------------------------------

        r"οστεοαρθρ.{0,150}"
        r"εσω.{0,50}διαμερισ",

        r"εσω.{0,50}διαμερισ.{0,150}"
        r"οστεοαρθρ",


        # ----------------------------------------------
        # Turco: cartílago articular normal
        # ----------------------------------------------

        r"eklem k[iı]k[iı]rdaklar[iı].{0,120}"
        r"(normal|korunmus)",


        # ----------------------------------------------
        # Búlgaro
        # ----------------------------------------------

        r"хрущял.{0,160}"
        r"(съхранен|интактен)",
        # Croata - caso 3051
        r"medijalnog kompartmenta.{0,180}"
        r"oa promjen",

        r"oa promjen.{0,200}"
        r"hondromalacija\s*(ii|2).{0,100}"
        r"lateralnog tibijalnog platoa",

        # ----------------------------------------------
        # Inglés
        # ----------------------------------------------

        r"degenerative changes\s*:\s*none",

        r"\bno degenerative changes\b",


        # ----------------------------------------------
        # Neerlandés
        # "Normaal voorkomen kraakbeen"
        # = cartílago de aspecto normal
        # ----------------------------------------------

        r"normaal voorkomen kraakbeen",


        # ----------------------------------------------
        # Búlgaro
        # cartílago lateral con espesor/congruencia
        # preservados
        # ----------------------------------------------

        r"ставният хрущял.{0,220}"
        r"латералния кондил.{0,160}"
        r"запазена дебелина и конгруентност",

        r"латералния кондил.{0,160}"
        r"запазена дебелина и конгруентност"

    ]


    # ==================================================
    # 3. AMBIGUOS
    # No implican OA lateral por sí solos
    # ==================================================

    ambiguos = [

        # Lesiones focales
        r"focal.{0,70}(chondrosis|chondral)",

        r"focal osteochondral injury",

        r"osteochondral injury",

        # Alteraciones leves
        r"low[- ]grade cartilage fissur",

        # Trauma / fractura
        r"subchondral fracture",

        r"osteochondral impaction fracture",

        # Importante:
        # "tricompartmental chondrosis" no lo tratamos
        # automáticamente como Lateral OA
        r"tricompartmental chondrosis"
    ]


    # ==================================================
    # 4. BUSCAR EVIDENCIA
    # ==================================================

    if contiene(texto, positivos_claros):
        evidencia_positiva = True

    if contiene(texto, negativos_claros):
        evidencia_negativa = True

    if contiene(texto, ambiguos):
        evidencia_ambigua = True


    # ==================================================
    # 5. DECISIÓN FINAL
    # ==================================================

    # La evidencia positiva específica tiene prioridad.
    # Esto evita que un reporte con OA medial + lateral
    # termine clasificado como negativo.
    if evidencia_positiva:
        return 1

    if evidencia_negativa:
        return 0

    if evidencia_ambigua:
        return -1

    return -1

In [28]:
label_actual = "Lateral OA"
columna_pred = "Lateral_OA_pred"

gold_dev[columna_pred] = (
    gold_dev["Report"]
    .apply(extraer_Lateral_OA)
)

print(
    f"=== EVALUACIÓN {label_actual.upper()} ==="
)

evaluar_extractor(
    gold_dev,
    label_actual,
    columna_pred
)

print("\nDistribución:")
print(
    gold_dev[columna_pred]
    .value_counts()
)

=== EVALUACIÓN LATERAL OA ===
Casos evaluados (DEV): 40
Casos resueltos (Cobertura 62.5%): 25
Aciertos: 25 | Errores: 0
Accuracy: 1.0
Precision: 1.0 | Recall: 1.0 | F1: 1.0

Distribución:
Lateral_OA_pred
 0    19
-1    15
 1     6
Name: count, dtype: int64


#PF OA (Patellofemoral OA)

In [29]:
# @title
# --- EXTRACTOR PF OA V7 ---

def extraer_PF_OA(reporte):

    texto = normalizar_texto(reporte)

    evidencia_positiva = False
    evidencia_negativa = False
    evidencia_leve = False


    # ==================================================
    # 1. POSITIVOS FUERTES
    # ==================================================

    positivos_fuertes = [

        # Diagnóstico explícito
        r"patellofemoral osteoarthritis",
        r"patellofemoral osteoarthrosis",
        r"patellofemoral oa",

        r"oa patelofemoral",
        r"osteoartrosis patelofemoral",
        r"artrosis patelofemoral",


        # Tricompartimental
        r"tricompartmental.{0,80}"
        r"(osteoarthritis|osteoarthrosis|\boa\b)",

        r"oa of all three compart\w*",

        r"tricompartmental chondrosis",


        # ----------------------------------------------
        # PATELA
        # ----------------------------------------------

        r"patellar cartilage.{0,80}"
        r"(high[- ]grade|full[- ]thickness|severe|complete loss)",

        r"patellar facet.{0,80}"
        r"(high[- ]grade|full[- ]thickness|deep fissur|delaminating fissur)",

        r"(high[- ]grade|full[- ]thickness)"
        r".{0,50}"
        r"(patellar cartilage|patellar facet)",


        # ----------------------------------------------
        # TRÓCLEA
        # ----------------------------------------------

        r"trochlear cartilage.{0,80}"
        r"(high[- ]grade|full[- ]thickness|deep fissur|severe)",

        r"trochlea.{0,80}"
        r"(high[- ]grade|full[- ]thickness|deep fissur|severe)",

        r"(high[- ]grade|full[- ]thickness)"
        r".{0,50}"
        r"(trochlea|trochlear cartilage)",


        # ----------------------------------------------
        # Condromalacia grado 3 / 4
        # ----------------------------------------------

        r"grade\s*(?:3|4|iii|iv).{0,80}"
        r"chondromalacia.{0,80}"
        r"(trochlea|patella|patellar)",

        r"chondromalacia.{0,80}"
        r"grade\s*(?:3|4|iii|iv)",

        r"grade\s*(?:3|4|iii|iv).{0,80}"
        r"(trochlea|patella|patellar)",


        # ----------------------------------------------
        # Patela / tróclea significativa
        # ----------------------------------------------

        r"patellar facet.{0,180}"
        r"(deep|delaminating).{0,80}fissur",

        r"femoral trochlea.{0,120}"
        r"moderate thinning",

        r"tricompartmental moderate chondral thinning"
        r".{0,180}patellofemoral",


        # ----------------------------------------------
        # Español
        # ----------------------------------------------

        r"condropatia.{0,100}"
        r"grado\s*(?:3|4|iii|iv)"
        r".{0,120}"
        r"(rotula|troclea)",

        r"(rotula|troclea).{0,120}"
        r"condropatia.{0,100}"
        r"grado\s*(?:3|4|iii|iv)",

        r"condropatia focal.{0,80}"
        r"grado\s*(?:3|4|iii|iv)"
        r".{0,120}"
        r"troclea",

        r"ulcera condral.{0,120}"
        r"espesor total.{0,120}"
        r"(rotula|troclea)",

        r"ulceras condrales.{0,160}"
        r"espesor total.{0,160}"
        r"(rotula|troclea)",

        r"ulceras condrales.{0,160}"
        r"espesor total.{0,100}"
        r"distribucion difusa.{0,80}"
        r"rotula",

        r"condropatia.{0,100}"
        r"grado\s*2.{0,30}(?:y|,).{0,30}3"
        r".{0,100}rotula"
    ]


    # ==================================================
    # 2. NEGATIVOS FUERTES
    # ==================================================

    negativos_fuertes = [

        # Inglés
        r"patellofemoral joint.{0,180}"
        r"without chondral lesion",

        r"patellar and trochlear cartilage.{0,120}"
        r"(normal|intact|preserved|congruent)",

        r"patellofemoral compartment cartilage.{0,120}"
        r"(normal|intact|preserved)",

        r"no cartilage defects?.{0,80}"
        r"(in )?(the )?trochlea",

        r"tibiofemoral and patellofemoral"
        r".{0,80}articular cartilage.{0,40}"
        r"(is )?normal",


        # NUEVO V7: caso 3556
        r"\bcartilages?\s+normal\b",


        # Español
        r"cartilagos? patelofemorales?.{0,100}"
        r"sin alteraciones",

        r"compartimento patelofemoral.{0,120}"
        r"sin alteraciones",

        r"cartilagos de los compartimentos"
        r".{0,120}patelofemorales"
        r".{0,80}sin alteraciones",

        r"resto de los cartilagos"
        r".{0,120}patelofemorales"
        r".{0,80}sin alteraciones",

        r"cartilagos.{0,150}"
        r"femorotibiales y patelofemorales"
        r".{0,80}sin alteraciones",


        # Turco
        r"eklem k[iı]k[iı]rdaklar[iı]"
        r".{0,100}normal",

        r"femoral troklear oluk.{0,120}"
        r"patellofemoral.{0,100}normal",


        # Griego
        r"προσθι.{0,40}διαμερισ"
        r".{0,120}δεν.{0,80}παθολογ",


        # NUEVO V7: neerlandés, caso 2982
        r"normaal voorkomen kraakbeen",

        r"kraakbeen.{0,50}"
        r"(normaal|intact)"
    ]


    # ==================================================
    # 3. CAMBIOS LEVES
    # Si no existe evidencia positiva fuerte -> 0
    # ==================================================

    cambios_leves = [

        # Mild PF
        r"mild patellofemoral chondrosis",

        r"scattered foci of mild chondrosis"
        r".{0,100}patellofemoral",

        # Low-grade
        r"focal.{0,50}low[- ]grade.{0,50}"
        r"chondrosis.{0,80}patellar facet",

        r"focal grade.{0,80}"
        r"chondrosis.{0,100}patellar facet",

        r"low[- ]grade articular cartilage loss"
        r".{0,100}patellar",

        r"slight chondrosis.{0,60}patella",


        # Croata grado 1
        r"hondromalaci.{0,80}"
        r"(1\.?\s*stupnja|grade\s*1)"
        r".{0,120}patel",

        # NUEVO V7: caso 3051
        r"hondromalaci.{0,80}"
        r"(i°|i\s*°|1°)"
        r".{0,120}patel"
    ]


    # ==================================================
    # 4. BUSCAR EVIDENCIA
    # ==================================================

    if contiene(texto, positivos_fuertes):
        evidencia_positiva = True

    if contiene(texto, negativos_fuertes):
        evidencia_negativa = True

    if contiene(texto, cambios_leves):
        evidencia_leve = True


    # ==================================================
    # 5. DECISIÓN FINAL
    # ==================================================

    # Evidencia fuerte de PF OA tiene prioridad
    if evidencia_positiva:
        return 1

    if evidencia_negativa:
        return 0

    # Cambios grado I / mild / low-grade:
    # según el GOLD no alcanzan el umbral de PF OA
    if evidencia_leve:
        return 0

    return -1

In [30]:
label_actual = "PF OA"
columna_pred = "PF_OA_pred"

gold_dev[columna_pred] = (
    gold_dev["Report"]
    .apply(extraer_PF_OA)
)

print(
    f"=== EVALUACIÓN {label_actual.upper()} ==="
)

evaluar_extractor(
    gold_dev,
    label_actual,
    columna_pred
)

print("\nDistribución:")
print(
    gold_dev[columna_pred]
    .value_counts()
)

=== EVALUACIÓN PF OA ===
Casos evaluados (DEV): 40
Casos resueltos (Cobertura 70.0%): 28
Aciertos: 28 | Errores: 0
Accuracy: 1.0
Precision: 1.0 | Recall: 1.0 | F1: 1.0

Distribución:
PF_OA_pred
 0    18
-1    12
 1    10
Name: count, dtype: int64


#Effusion (derrame articular)

In [31]:
# @title
# --- EXTRACTOR EFFUSION V4 ---

def extraer_Effusion(reporte):

    texto = normalizar_texto(reporte)


    # ==================================================
    # 1. NEGATIVOS EXPLÍCITOS
    # ==================================================

    negativos_fuertes = [

        # Inglés
        r"\bno knee effusion\b",
        r"\bno joint effusion\b",
        r"\bno effusion\b",

        r"no evidence of.{0,40}effusion",
        r"no significant.{0,30}effusion",
        r"without.{0,30}effusion",
        r"effusion.{0,30}absent",

        # Español
        r"\bno hay derrame\b",
        r"\bsin derrame\b",
        r"no se observa.{0,40}derrame",

        # Alemán
        r"kein gelenkerguss"
    ]


    # ==================================================
    # 2. CASOS AMBIGUOS
    # GOLD inconsistente → abstención
    # ==================================================

    ambiguos = [

        # Español
        r"\bleve derrame articular\b",
        r"\bderrame articular leve\b",

        # Inglés
        r"\bminimal amount of.{0,40}effusion\b",

        r"\bminimal amount of.{0,30}"
        r"(right |left )?knee effusion\b",

        # Búlgaro
        r"минимален.{0,50}ставен излив",
        r"минимално.{0,50}ставен излив"
    ]


    # ==================================================
    # 3. POSITIVOS FUERTES
    # ==================================================

    positivos_fuertes = [

        # -------------------------
        # Inglés
        # -------------------------

        r"\bmoderate.{0,40}(joint|knee)?.{0,20}effusion\b",

        r"\blarge.{0,40}(joint|knee)?.{0,20}effusion\b",

        r"\bsmall.{0,40}(joint|knee)?.{0,20}effusion\b",

        r"\bmild.{0,40}(joint|knee)?.{0,20}effusion\b",

        r"\bjoint effusion\b",
        r"\bknee effusion\b",

        r"\beffusion is present\b",
        r"\beffusion present\b",

        # Caso 2781
        r"\beffusion with synovitis\b",

        # typo observado
        r"\beffsuion\b",


        # -------------------------
        # Español
        # -------------------------

        r"\bleve a moderado derrame articular\b",

        r"\bmoderado derrame articular\b",
        r"\bderrame articular moderado\b",

        r"\bimportante derrame\b",
        r"\babundante derrame\b",

        r"\bderrame articular\b",

        # Caso 4043
        r"\bderrame con sinovitis\b",


        # -------------------------
        # Turco
        # -------------------------

        r"diz eklem.{0,100}sivi artisi",

        r"eklem ici.{0,100}sivi",

        r"eklem.{0,100}efüzyon",

        r"efüzyon mevcuttur",

        # Caso 1352
        r"diz eklemi ici.{0,80}"
        r"sivi miktari.{0,80}"
        r"artmis",

        # Caso 3147
        r"diz eklem.{0,100}"
        r"artmis efüzyon",

        r"artmis efüzyon mevcuttur",


        # -------------------------
        # Croata
        # -------------------------

        r"izljev.{0,60}zglob",
        r"izljeva.{0,60}zglob",


        # -------------------------
        # Griego
        # -------------------------

        r"συλλογ.{0,50}υγρ.{0,100}ενδαρθρ",

        r"ποσοτητα υγρου.{0,100}ενδαρθρ",


        # -------------------------
        # Búlgaro
        # -------------------------

        r"ставен излив",
        r"излив.{0,50}став",


        # -------------------------
        # Alemán
        # -------------------------

        r"gelenkerguss",
        r"erguss.{0,50}gelenk",


        # -------------------------
        # Neerlandés
        # -------------------------

        r"gewrichtseffusie",
        r"gewrichtsvocht",

        # Caso 2982
        r"matige opzetting.{0,80}"
        r"suprapatellaire recessus"
    ]


    # ==================================================
    # 4. DECISIÓN
    # ==================================================

    # Ausencia explícita
    if contiene(texto, negativos_fuertes):
        return 0

    # Casos realmente ambiguos según GOLD
    if contiene(texto, ambiguos):
        return -1

    # Presencia clara
    if contiene(texto, positivos_fuertes):
        return 1

    return -1

In [32]:
label_actual = "Effusion"
columna_pred = "Effusion_pred"

gold_dev[columna_pred] = (
    gold_dev["Report"]
    .apply(extraer_Effusion)
)

evaluar_extractor(
    gold_dev,
    label_actual,
    columna_pred
)

print("\nDistribución:")
print(
    gold_dev[columna_pred]
    .value_counts()
)

Casos evaluados (DEV): 40
Casos resueltos (Cobertura 77.5%): 31
Aciertos: 30 | Errores: 1
Accuracy: 0.968
Precision: 0.962 | Recall: 1.0 | F1: 0.98

Distribución:
Effusion_pred
 1    26
-1     9
 0     5
Name: count, dtype: int64


#Synovitis

In [33]:
# @title
# --- EXTRACTOR SYNOVITIS V2 ---

def extraer_Synovitis(reporte):

    texto = normalizar_texto(reporte)


    # ==================================================
    # 1. POSITIVOS FUERTES
    # ==================================================

    positivos_fuertes = [

        # Inglés
        r"\bsynovitis\b",
        r"synovial thickening",
        r"thickening.{0,40}synovium",
        r"hypertrophy.{0,40}synovial",
        r"hypertrophy.{0,40}synovium",
        r"reactive synovitis",
        r"chronic reactive synovitis",

        # Español
        r"\bsinovitis\b",
        r"engrosamiento sinovial",
        r"engrosamiento.{0,40}sinovial",
        r"hipertrofia.{0,40}sinovial",

        # Turco
        r"\bsinovit\b",
        r"sinovyal.{0,40}"
        r"(kalinlasma|hipertrofi)",

        # Croata
        r"\bsinovitis\b",
        r"sinovijal.{0,40}"
        r"(zadebljan|hipertrof)",

        # Griego
        r"υμενιτιδ",
        r"αρθρικ.{0,30}υμεν.{0,50}"
        r"(παχυν|υπερτροφ)",

        # Búlgaro
        r"синовит",
        r"синовиал.{0,50}"
        r"(удебел|хипертроф)",

        # Alemán
        r"\bsynovitis\b",
        r"synovial.{0,40}"
        r"(verdick|hypertroph)",

        # Neerlandés
        r"\bsynovitis\b",
        r"synoviaal.{0,40}"
        r"(verdikt|hypertrof)",
                # ----------------------------------------------
        # Turco
        # Casos 1816 y 2472
        #
        # "diz eklemi içerisinde yaygın sıvı artışı"
        # = aumento difuso de líquido intraarticular
        # ----------------------------------------------

        r"diz eklemi.{0,50}"
        r"icerisinde.{0,50}"
        r"yayg[iı]n.{0,30}"
        r"s[iı]v[iı].{0,30}"
        r"art[iı]s",
        # Variante por posibles diferencias
        # de normalización
        r"diz eklemi icerisinde yaygin sivi artisi"
    ]


    # ==================================================
    # 2. NEGATIVOS EXPLÍCITOS DE SYNOVITIS
    # ==================================================

    negativos_fuertes = [

        # Inglés
        r"\bno synovitis\b",
        r"\bwithout synovitis\b",
        r"no evidence of.{0,40}synovitis",

        r"\bminimal amount of.{0,50}effusion\b",
        r"\bminimal.{0,30}joint effusion\b",
        r"\bminimal.{0,30}knee effusion\b",

        r"no synovial thickening",
        r"synovium.{0,30}(normal|unremarkable)",

        # Español
        r"\bsin sinovitis\b",
        r"\bno hay sinovitis\b",
        r"no se observa.{0,40}sinovitis",

        r"sin engrosamiento sinovial",

        # Alemán
        r"\bkeine synovitis\b",

        r"synovitis.{0,20}absent",

        # ----------------------------------------------
        # Búlgaro
        # "минимален ставен излив"
        # = derrame articular mínimo
        # ----------------------------------------------

        r"минимален.{0,30}ставен излив",

        r"минимално.{0,30}ставен излив"
    ]


    # ==================================================
    # 3. PROXY NEGATIVO:
    # AUSENCIA EXPLÍCITA DE DERRAME
    # ==================================================

    sin_derrame = [

        # Inglés
        r"\bno knee effusion\b",
        r"\bno joint effusion\b",
        r"\bno significant knee effusion\b",
        r"\bno evidence of knee effusion\b",

        # Español
        r"\bno hay derrame articular\b",
        r"\bsin derrame articular\b",

        # Alemán
        r"\bkein gelenkerguss\b",

        # Variantes genéricas
        r"\beffusion absent\b"
    ]


    # ==================================================
    # 4. DECISIÓN
    # ==================================================

    # Una mención explícita de sinovitis manda
    if contiene(texto, positivos_fuertes):
        return 1

    # Negación explícita
    if contiene(texto, negativos_fuertes):
        return 0

    # Proxy negativo conservador
    if contiene(texto, sin_derrame):
        return 0

    return -1

In [34]:
label_actual = "Synovitis"
columna_pred = "Synovitis_pred"

gold_dev[columna_pred] = (
    gold_dev["Report"]
    .apply(extraer_Synovitis)
)

print(
    f"=== EVALUACIÓN {label_actual.upper()} ==="
)

evaluar_extractor(
    gold_dev,
    label_actual,
    columna_pred
)

print("\nDistribución:")
print(
    gold_dev[columna_pred]
    .value_counts()
)

=== EVALUACIÓN SYNOVITIS ===
Casos evaluados (DEV): 40
Casos resueltos (Cobertura 47.5%): 19
Aciertos: 18 | Errores: 1
Accuracy: 0.947
Precision: 0.909 | Recall: 1.0 | F1: 0.952

Distribución:
Synovitis_pred
-1    21
 1    11
 0     8
Name: count, dtype: int64


#Baker's cyst. (quiste de Baker)

In [35]:
# @title
# --- EXTRACTOR BAKER'S CYST V2 ---

def extraer_Bakers(reporte):

    texto = normalizar_texto(reporte)


    # ==================================================
    # 1. NEGATIVOS EXPLÍCITOS
    # ==================================================

    negativos_fuertes = [

        # Inglés
        r"\bno baker'?s cyst\b",
        r"\bno bakers cyst\b",
        r"\bno popliteal cyst\b",

        r"without.{0,30}baker'?s cyst",
        r"without.{0,30}popliteal cyst",

        r"baker'?s? cyst.{0,20}(none|absent)",

        # NUEVO: caso 3064
        r"baker cyst\s*:\s*none",

        # Español
        r"\bno hay quistes? popliteos?\b",
        r"\bsin quistes? popliteos?\b",
        r"\bno hay quiste popliteo\b",
        r"\bsin quiste popliteo\b",

        # Alemán
        r"\bkeine baker[- ]zyste\b",

        # Genéricos
        r"popliteal cyst.{0,20}absent",

        r"\bno knee effusion is present\b",

        r"\bno knee effusion\b",

        r"\bno joint effusion\b",

        r"\bno evidence of knee effusion\b"
    ]


    # ==================================================
    # 2. MENCIONES PEQUEÑAS / DUDOSAS
    # No las convertimos en negativo.
    # Simplemente nos abstenemos.
    # ==================================================

    ambiguos = [

        # Inglés
        r"\btrace baker'?s cyst\b",

        r"\bsmall baker'?s cyst\b",

        r"\bsmall popliteal cyst\b",

        r"\btrace popliteal cyst\b",


        # Español
        r"\bpequeno quiste popliteo\b",

        r"\bpequenos quistes popliteos\b",

        r"\bquiste popliteo pequeno\b",


        # Griego
        # μικρή = pequeña
        r"μικρ.{0,80}baker",

        r"μικρ.{0,80}κυστη.{0,40}baker"
    ]


    # ==================================================
    # 3. POSITIVOS
    # ==================================================

    positivos_fuertes = [

        # -------------------------
        # Inglés
        # -------------------------

        r"\bbaker'?s cyst\b",
        r"\bbakers cyst\b",
        r"\bpopliteal cyst\b",

        r"\bmoderate[- ]sized popliteal cyst\b",
        r"\blarge baker'?s cyst\b",
        r"\blarge popliteal cyst\b",


        # -------------------------
        # Español
        # -------------------------

        r"\bquiste de baker\b",
        r"\bquiste baker\b",
        r"\bquiste popliteo\b",
        r"\bquistes popliteos\b",


        # -------------------------
        # Turco
        # -------------------------

        r"\bbaker kisti\b",
        r"\bpopliteal kist\b",


        # -------------------------
        # Croata
        # -------------------------

        r"\bbaker cista\b",
        r"\bbakerova cista\b",


        # -------------------------
        # Alemán
        # -------------------------

        r"\bbaker[- ]zyste\b",


        # -------------------------
        # Neerlandés
        # -------------------------

        r"\bbaker.?cyste\b",
        r"\bpopliteale cyste\b",


        # -------------------------
        # Griego
        # -------------------------

        r"κυστη.{0,20}baker",
        r"baker.{0,20}κυστη",


        # -------------------------
        # Búlgaro
        # -------------------------

        r"бекерова киста",
        r"киста.{0,20}бекер"
    ]


    # ==================================================
    # 4. DECISIÓN
    # ==================================================

    # Ausencia explícita
    if contiene(texto, negativos_fuertes):
        return 0

    # Mención demasiado pequeña/dudosa
    if contiene(texto, ambiguos):
        return -1

    # Presencia explícita
    if contiene(texto, positivos_fuertes):
        return 1

    return -1

In [36]:
label_actual = "Baker's"
columna_pred = "Bakers_pred"

gold_dev[columna_pred] = (
    gold_dev["Report"]
    .apply(extraer_Bakers)
)

print(
    f"=== EVALUACIÓN {label_actual.upper()} ==="
)

evaluar_extractor(
    gold_dev,
    label_actual,
    columna_pred
)

print("\nDistribución:")
print(
    gold_dev[columna_pred]
    .value_counts()
)

=== EVALUACIÓN BAKER'S ===
Casos evaluados (DEV): 40
Casos resueltos (Cobertura 47.5%): 19
Aciertos: 17 | Errores: 2
Accuracy: 0.895
Precision: 0.778 | Recall: 1.0 | F1: 0.875

Distribución:
Bakers_pred
-1    21
 0    10
 1     9
Name: count, dtype: int64


#Contusion

In [37]:
# @title
# --- EXTRACTOR CONTUSION V2 ---

def extraer_Contusion(reporte):

    texto = normalizar_texto(reporte)


    # ==================================================
    # 1. NEGACIONES EXPLÍCITAS DE CONTUSIÓN
    # Máxima prioridad
    # ==================================================

    negativos_explicitos = [

        # Inglés
        r"\bno bone contusion\b",
        r"\bno bone bruise\b",
        r"\bno osseous contusion\b",

        # Casos 470 / 906
        r"\bno fracture or bone contusion\b",
        r"\bno fracture or bone bruise\b",

        r"no evidence of.{0,40}bone contusion",
        r"no evidence of.{0,40}bone bruise",

        r"without.{0,30}bone contusion",
        r"without.{0,30}bone bruise",

        # Español
        r"\bsin contusion osea\b",
        r"\bno hay contusion osea\b",
        r"no se observa.{0,30}contusion osea",

        # Alemán
        r"\bkeine knochenkontusion\b"
    ]


    # ==================================================
    # 2. POSITIVOS FUERTES
    # ==================================================

    positivos_fuertes = [

        # -------------------------
        # Inglés
        # -------------------------

        r"\bbone contusion\b",
        r"\bbony contusion\b",
        r"\bosseous contusion\b",

        r"\bbone bruise\b",
        r"\bbone bruising\b",

        r"contusional.{0,50}"
        r"(bone marrow edema|marrow edema)",

        r"bone marrow edema.{0,60}"
        r"compatible with.{0,30}contusion",


        # -------------------------
        # Español
        # -------------------------

        r"\bcontusion osea\b",

        r"\bedema oseo contusional\b",

        r"edema oseo.{0,50}"
        r"compatible con.{0,30}contusion",


        # -------------------------
        # Turco
        # -------------------------

        r"kemik kontuzyon",

        r"kontuzyonel.{0,50}"
        r"kemik iligi odem",


        # -------------------------
        # Croata
        # -------------------------

        r"kostan.{0,20}kontuz",
        r"koštan.{0,20}kontuz",

        r"kontuzij.{0,50}"
        r"(kostan|koštan).{0,30}edem",


        # -------------------------
        # Alemán
        # -------------------------

        r"\bknochenkontusion\b",
        r"\bbone bruise\b",


        # -------------------------
        # Neerlandés
        # -------------------------

        r"\bbotcontusie\b",
        r"\bbotcontusies\b",


        # -------------------------
        # Búlgaro
        # -------------------------

        # Caso 2885
        r"контузионен.{0,40}"
        r"костно[- ]мозъчен едем",

        r"контузионен.{0,60}"
        r"костномозъчен едем",

        # Caso 3470
        r"травматичен.{0,40}"
        r"костно[- ]мозъчен едем",

        r"травматичен.{0,60}"
        r"костномозъчен едем",

        r"костна контуз",


        # -------------------------
        # Griego
        # -------------------------

        r"οστικ.{0,30}θλασ"
    ]


    # ==================================================
    # 3. NORMALIDAD ÓSEA CLARA
    # Solo se usa si NO encontramos contusión explícita
    # ==================================================

    negativos_normalidad = [

        # Inglés
        r"\bbones\s*:\s*normal\b",

        r"\bnormal visualized bones\b",

        r"background marrow signal.{0,40}normal",

        r"no suspicious focal marrow signal alteration",


        # Español
        r"no hay alteraciones de senal significativas"
        r".{0,40}medula osea"
    ]

    positivos_otras_contusiones = [

        # Inglés
        r"\bmuscle contusion\b",
        r"\bmuscular contusion\b",

        r"\bpopliteus muscle contusion\b",

        # Español
        r"\bcontusion muscular\b",

        # Variantes
        r"muscle.{0,30}contusion",

        r"bone marrow edema.{0,120}"
        r"without evidence of associated fracture",

        r"marrow edema.{0,120}"
        r"without evidence of associated fracture"
    ]


    # ==================================================
    # 4. DECISIÓN
    # ==================================================

    # Contusión muscular explícita:
    # caso 533
    if contiene(texto, positivos_otras_contusiones):
        return 1

    # Negación explícita de contusión ósea
    if contiene(texto, negativos_explicitos):
        return 0

    # Contusión ósea explícita
    if contiene(texto, positivos_fuertes):
        return 1

    # Normalidad ósea
    if contiene(texto, negativos_normalidad):
        return 0

    return -1

In [38]:
label_actual = "Contusion"
columna_pred = "Contusion_pred"

gold_dev[columna_pred] = (
    gold_dev["Report"]
    .apply(extraer_Contusion)
)

print(
    f"=== EVALUACIÓN {label_actual.upper()} ==="
)

evaluar_extractor(
    gold_dev,
    label_actual,
    columna_pred
)

print("\nDistribución:")
print(
    gold_dev[columna_pred]
    .value_counts()
)

=== EVALUACIÓN CONTUSION ===
Casos evaluados (DEV): 40
Casos resueltos (Cobertura 52.5%): 21
Aciertos: 18 | Errores: 3
Accuracy: 0.857
Precision: 0.769 | Recall: 1.0 | F1: 0.87

Distribución:
Contusion_pred
-1    19
 1    13
 0     8
Name: count, dtype: int64


#Fracture

In [39]:
# @title
# --- EXTRACTOR FRACTURE V2 ---

def extraer_Fracture(reporte):

    texto = normalizar_texto(reporte)


    # ==================================================
    # 1. POSITIVOS FUERTES
    # ==================================================

    positivos_fuertes = [

        # Inglés
        r"\bsubchondral fracture\b",
        r"\binsufficiency fracture\b",
        r"\bimpaction fracture\b",
        r"\bosteochondral fracture\b",
        r"\bosteochondral impaction fracture\b",
        r"\bavulsion fracture\b",
        r"\bhairline fracture\b",
        r"\bsegond fracture\b",
        r"\btibial plateau fracture\b",
        r"\bfemoral condyle fracture\b",
        r"\bfracture line\b",

        # Español
        r"\bfractura subcondral\b",
        r"\bfractura osteocondral\b",
        r"\bfractura por insuficiencia\b",
        r"\bfractura de impactacion\b",
        r"\bfractura por impactacion\b",
        r"\bfractura por avulsion\b",
        r"\blinea de fractura\b",

        # Turco
        r"\bsubartikuler fraktur\b",
        r"\bsubkondral fraktur\b",
        r"\bk[iı]r[iı]k hatt[iı]\b",
        r"\bsubkondral k[iı]r[iı]k\b",
        # Turco
        # Caso 1352:
        # "subkondral kırığı"
        r"subkondral.{0,20}k[iı]r[iı][gğ]",

        r"subkondral.{0,20}k[iı]r[iı]k",

        r"k[iı]r[iı][gğ][iı].{0,40}"
        r"kemik iligi odem",

        # Croata
        r"impresijsk.{0,30}fraktur",
        r"subhondral.{0,30}fraktur",

        # Alemán
        r"\bsubchondrale fraktur\b",
        r"\bimpaktionsfraktur\b",

        # Neerlandés
        r"\bimpactiefractuur\b",
        r"\bsubchondrale fractuur\b",

        # Búlgaro
        r"\bостеохондрална фрактура\b",
        r"\bсубхондрална фрактура\b",

        # Griego
        r"οστεοχονδρ.{0,40}καταγμα",
        r"ανεπαρκ.{0,40}καταγμα"
    ]


    # ==================================================
    # 2. POSITIVOS GENÉRICOS
    # ==================================================

    positivos_genericos = [

        r"\bfracture\b",
        r"\bfractures\b",

        r"\bfractura\b",
        r"\bfracturas\b",

        r"\bfraktur\b",

        r"\bfraktura\b",

        r"\bfractuur\b",

        r"\bфрактура\b",

        r"καταγμα"
    ]


    # ==================================================
    # 3. NEGATIVOS CLAROS
    # ==================================================

    negativos_fuertes = [

        # Inglés
        r"\bno fracture\b",
        r"\bno fractures\b",
        r"\bno acute fracture\b",
        r"no evidence of.{0,40}fracture",
        r"without.{0,30}fracture",

        # Inglés
        r"old fracture", r"healed fracture", r"previous fracture", r"history of fracture",
        # Español
        r"fractura antigua", r"fractura consolidada", r"fractura previa", r"antecedente de fractura",
        # Otros idiomas comunes en el dataset
        r"alte fraktur", r"stara fraktura", r"stare frakture",

        # Español
        r"\bsin fractura\b",
        r"\bsin fracturas\b",
        r"\bno hay fractura\b",
        r"\bno hay fracturas\b",
        r"no se observa.{0,40}fractura",

        # Alemán
        r"\bkeine fraktur\b",

        # Neerlandés
        r"\bgeen fractuur\b",

        # Croata
        r"bez znakova.{0,40}fraktur",
        r"nema.{0,30}fraktur",

        # Inglés
        # Caso 656
        r"\bbones\s*:\s*normal\b",

        # Caso 4187
        r"\bnormal visualized bones\b"
    ]


    # ==================================================
    # 4. DECISIÓN
    # ==================================================

    # Una fractura específica y explícita manda.
    # Esto corrige 2981 y 3064.
    if contiene(texto, positivos_fuertes):
        return 1

    # Si solo tenemos una negación explícita
    if contiene(texto, negativos_fuertes):
        return 0

    # Fractura genérica no negada
    if contiene(texto, positivos_genericos):
        return 1

    return -1

In [40]:
label_actual = "Fracture"
columna_pred = "Fracture_pred"

gold_dev[columna_pred] = (
    gold_dev["Report"]
    .apply(extraer_Fracture)
)

print(
    f"=== EVALUACIÓN {label_actual.upper()}  ==="
)

evaluar_extractor(
    gold_dev,
    label_actual,
    columna_pred
)

print("\nDistribución:")
print(
    gold_dev[columna_pred]
    .value_counts()
)

=== EVALUACIÓN FRACTURE  ===
Casos evaluados (DEV): 40
Casos resueltos (Cobertura 55.0%): 22
Aciertos: 16 | Errores: 6
Accuracy: 0.727
Precision: 0.625 | Recall: 1.0 | F1: 0.769

Distribución:
Fracture_pred
-1    18
 1    16
 0     6
Name: count, dtype: int64


## Resumen final de Weak Labeling sobre DEV

In [41]:
# ============================================================
# RESUMEN FINAL DE WEAK LABELING - GOLD DEV
# ============================================================

from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd


# Relación entre etiqueta GOLD y columna predicha
pred_cols = {
    "ACL": "ACL_pred",
    "MCL": "MCL_pred",
    "Medial Meniscus": "Medial_Meniscus_pred",
    "Lateral Meniscus": "Lateral_Meniscus_pred",
    "Medial OA": "Medial_OA_pred",
    "Lateral OA": "Lateral_OA_pred",
    "PF OA": "PF_OA_pred",
    "Effusion": "Effusion_pred",
    "Synovitis": "Synovitis_pred",
    "Baker's": "Bakers_pred",
    "Contusion": "Contusion_pred",
    "Fracture": "Fracture_pred"
}


resultados_dev = []


for label, pred_col in pred_cols.items():

    # Solo casos donde el weak labeler tomó una decisión
    resueltos = gold_dev[
        gold_dev[pred_col] != -1
    ].copy()

    total_gold = len(gold_dev)
    total_resueltos = len(resueltos)

    coverage = (
        total_resueltos / total_gold
        if total_gold > 0
        else 0
    )


    # Si no hubiera ningún caso resuelto
    if total_resueltos == 0:

        resultados_dev.append({
            "Label": label,
            "Total": total_gold,
            "Resueltos": 0,
            "Coverage (%)": 0,
            "Aciertos": 0,
            "Errores": 0,
            "Agreement (%)": 0,
            "Noise (%)": 0,
            "Precision": 0,
            "Recall": 0,
            "F1": 0
        })

        continue


    y_true = resueltos[label].astype(int)
    y_pred = resueltos[pred_col].astype(int)


    aciertos = (y_true == y_pred).sum()
    errores = total_resueltos - aciertos

    agreement = aciertos / total_resueltos

    # Ruido estimado entre las etiquetas resueltas
    noise = errores / total_resueltos


    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )


    resultados_dev.append({
        "Label": label,
        "Total": total_gold,
        "Resueltos": total_resueltos,
        "Coverage (%)": round(coverage * 100, 1),
        "Aciertos": aciertos,
        "Errores": errores,
        "Agreement (%)": round(agreement * 100, 1),
        "Noise (%)": round(noise * 100, 1),
        "Precision": round(precision, 3),
        "Recall": round(recall, 3),
        "F1": round(f1, 3)
    })


# Crear tabla
resumen_dev = pd.DataFrame(resultados_dev)

display(resumen_dev)

,Label,Total,Resueltos,Coverage (%),Aciertos,Errores,Agreement (%),Noise (%),Precision,Recall,F1
0,ACL,40,28,70.0,27,1,96.4,3.6,0.929,1.000,0.963
1,MCL,40,24,60.0,24,0,100.0,0.0,1.000,1.000,1.000
2,Medial Meniscus,40,27,67.5,25,2,92.6,7.4,0.895,1.000,0.944
3,Lateral Meniscus,40,24,60.0,22,2,91.7,8.3,0.923,0.923,0.923
4,Medial OA,40,29,72.5,27,2,93.1,6.9,0.818,1.000,0.900
5,Lateral OA,40,25,62.5,25,0,100.0,0.0,1.000,1.000,1.000
6,PF OA,40,28,70.0,28,0,100.0,0.0,1.000,1.000,1.000
7,Effusion,40,31,77.5,30,1,96.8,3.2,0.962,1.000,0.980
8,Synovitis,40,19,47.5,18,1,94.7,5.3,0.909,1.000,0.952
9,Baker's,40,19,47.5,17,2,89.5,10.5,0.778,1.000,0.875


#=== RESUMEN GLOBAL DEV ===

In [42]:
print("=== RESUMEN GLOBAL DEV ===")

print(
    f"Coverage promedio: "
    f"{resumen_dev['Coverage (%)'].mean():.1f}%"
)

print(
    f"Agreement promedio: "
    f"{resumen_dev['Agreement (%)'].mean():.1f}%"
)

print(
    f"Noise promedio: "
    f"{resumen_dev['Noise (%)'].mean():.1f}%"
)

print(
    f"F1 promedio: "
    f"{resumen_dev['F1'].mean():.3f}"
)

=== RESUMEN GLOBAL DEV ===
Coverage promedio: 61.9%
Agreement promedio: 92.8%
Noise promedio: 7.2%
F1 promedio: 0.931


## Evaluación final de Weak Labeling sobre GOLD HOLDOUT

In [43]:
# ============================================================
# APLICAR EXTRACTORES CONGELADOS AL GOLD HOLDOUT
# ============================================================

gold_holdout = gold_holdout.copy()

gold_holdout["ACL_pred"] = (
    gold_holdout["Report"].apply(extraer_acl)
)

gold_holdout["MCL_pred"] = (
    gold_holdout["Report"].apply(extraer_MCL)
)

gold_holdout["Medial_Meniscus_pred"] = (
    gold_holdout["Report"].apply(extraer_medial_meniscus)
)

gold_holdout["Lateral_Meniscus_pred"] = (
    gold_holdout["Report"].apply(extraer_lateral_meniscus)
)

gold_holdout["Medial_OA_pred"] = (
    gold_holdout["Report"].apply(extraer_Medial_OA)
)

gold_holdout["Lateral_OA_pred"] = (
    gold_holdout["Report"].apply(extraer_Lateral_OA)
)

gold_holdout["PF_OA_pred"] = (
    gold_holdout["Report"].apply(extraer_PF_OA)
)

gold_holdout["Effusion_pred"] = (
    gold_holdout["Report"].apply(extraer_Effusion)
)

gold_holdout["Synovitis_pred"] = (
    gold_holdout["Report"].apply(extraer_Synovitis)
)

gold_holdout["Bakers_pred"] = (
    gold_holdout["Report"].apply(extraer_Bakers)
)

gold_holdout["Contusion_pred"] = (
    gold_holdout["Report"].apply(extraer_Contusion)
)

gold_holdout["Fracture_pred"] = (
    gold_holdout["Report"].apply(extraer_Fracture)
)

print("✅ Extractores congelados aplicados al GOLD HOLDOUT")
print("Casos HOLDOUT:", len(gold_holdout))

✅ Extractores congelados aplicados al GOLD HOLDOUT
Casos HOLDOUT: 18


In [44]:
pred_cols = {
    "ACL": "ACL_pred",
    "MCL": "MCL_pred",
    "Medial Meniscus": "Medial_Meniscus_pred",
    "Lateral Meniscus": "Lateral_Meniscus_pred",
    "Medial OA": "Medial_OA_pred",
    "Lateral OA": "Lateral_OA_pred",
    "PF OA": "PF_OA_pred",
    "Effusion": "Effusion_pred",
    "Synovitis": "Synovitis_pred",
    "Baker's": "Bakers_pred",
    "Contusion": "Contusion_pred",
    "Fracture": "Fracture_pred"
}

In [45]:
# ============================================================
# RESUMEN FINAL - GOLD HOLDOUT
# ============================================================

resultados_holdout = []

for label, pred_col in pred_cols.items():

    resueltos = gold_holdout[
        gold_holdout[pred_col] != -1
    ].copy()

    total_gold = len(gold_holdout)
    total_resueltos = len(resueltos)

    coverage = (
        total_resueltos / total_gold
        if total_gold > 0
        else 0
    )

    if total_resueltos == 0:

        resultados_holdout.append({
            "Label": label,
            "Total": total_gold,
            "Resueltos": 0,
            "Coverage (%)": 0,
            "Aciertos": 0,
            "Errores": 0,
            "Agreement (%)": 0,
            "Noise (%)": 0,
            "Precision": 0,
            "Recall": 0,
            "F1": 0
        })

        continue

    y_true = resueltos[label].astype(int)
    y_pred = resueltos[pred_col].astype(int)

    aciertos = (y_true == y_pred).sum()
    errores = total_resueltos - aciertos

    agreement = aciertos / total_resueltos
    noise = errores / total_resueltos

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    resultados_holdout.append({
        "Label": label,
        "Total": total_gold,
        "Resueltos": total_resueltos,
        "Coverage (%)": round(coverage * 100, 1),
        "Aciertos": aciertos,
        "Errores": errores,
        "Agreement (%)": round(agreement * 100, 1),
        "Noise (%)": round(noise * 100, 1),
        "Precision": round(precision, 3),
        "Recall": round(recall, 3),
        "F1": round(f1, 3)
    })


resumen_holdout = pd.DataFrame(resultados_holdout)

display(resumen_holdout)

,Label,Total,Resueltos,Coverage (%),Aciertos,Errores,Agreement (%),Noise (%),Precision,Recall,F1
0,ACL,18,12,66.7,10,2,83.3,16.7,0.667,1.0,0.800
1,MCL,18,7,38.9,6,1,85.7,14.3,0.000,0.0,0.000
2,Medial Meniscus,18,7,38.9,6,1,85.7,14.3,0.750,1.0,0.857
3,Lateral Meniscus,18,5,27.8,3,2,60.0,40.0,0.500,1.0,0.667
4,Medial OA,18,5,27.8,4,1,80.0,20.0,1.000,0.5,0.667
5,Lateral OA,18,5,27.8,4,1,80.0,20.0,0.667,1.0,0.800
6,PF OA,18,5,27.8,3,2,60.0,40.0,0.500,0.5,0.500
7,Effusion,18,10,55.6,4,6,40.0,60.0,0.333,1.0,0.500
8,Synovitis,18,6,33.3,3,3,50.0,50.0,0.500,1.0,0.667
9,Baker's,18,4,22.2,4,0,100.0,0.0,1.000,1.0,1.000


In [46]:
print("=== RESUMEN GLOBAL HOLDOUT ===")

print(
    f"Coverage promedio: "
    f"{resumen_holdout['Coverage (%)'].mean():.1f}%"
)

print(
    f"Agreement promedio: "
    f"{resumen_holdout['Agreement (%)'].mean():.1f}%"
)

print(
    f"Noise promedio: "
    f"{resumen_holdout['Noise (%)'].mean():.1f}%"
)

print(
    f"F1 promedio: "
    f"{resumen_holdout['F1'].mean():.3f}"
)

=== RESUMEN GLOBAL HOLDOUT ===
Coverage promedio: 36.6%
Agreement promedio: 73.9%
Noise promedio: 26.1%
F1 promedio: 0.686


Los extractores mostraron buen desempeño en el conjunto de desarrollo, con agreement macro de 92.8% y ruido estimado de 7.2%. Sin embargo, al evaluarlos sin modificaciones sobre un conjunto holdout independiente de 18 estudios, el agreement macro disminuyó a 73.9% y el ruido aumentó a 26.1%, evidenciando una brecha de generalización y la dificultad de extraer etiquetas clínicas consistentes a partir de reportes multilingües. Debido al reducido tamaño del holdout, las métricas por etiqueta deben interpretarse con cautela, especialmente cuando el número de casos resueltos es bajo.

# 7. Generación de etiquetas débiles


In [47]:
# ============================================================
# APLICAR WEAK LABELERS A LOS 4349 ESTUDIOS SIN GOLD
# ============================================================

weak_df = weak_df.copy()

weak_df["ACL_weak"] = weak_df["Report"].apply(extraer_acl)
weak_df["MCL_weak"] = weak_df["Report"].apply(extraer_MCL)

weak_df["Medial_Meniscus_weak"] = (
    weak_df["Report"].apply(extraer_medial_meniscus)
)

weak_df["Lateral_Meniscus_weak"] = (
    weak_df["Report"].apply(extraer_lateral_meniscus)
)

weak_df["Medial_OA_weak"] = (
    weak_df["Report"].apply(extraer_Medial_OA)
)

weak_df["Lateral_OA_weak"] = (
    weak_df["Report"].apply(extraer_Lateral_OA)
)

weak_df["PF_OA_weak"] = (
    weak_df["Report"].apply(extraer_PF_OA)
)

weak_df["Effusion_weak"] = (
    weak_df["Report"].apply(extraer_Effusion)
)

weak_df["Synovitis_weak"] = (
    weak_df["Report"].apply(extraer_Synovitis)
)

weak_df["Bakers_weak"] = (
    weak_df["Report"].apply(extraer_Bakers)
)

weak_df["Contusion_weak"] = (
    weak_df["Report"].apply(extraer_Contusion)
)

weak_df["Fracture_weak"] = (
    weak_df["Report"].apply(extraer_Fracture)
)

print("✅ Weak labels generadas")
print("Estudios:", len(weak_df))

✅ Weak labels generadas
Estudios: 4349


In [48]:
# Añadir idioma al conjunto weak si todavía no existe
if "Idioma" not in weak_df.columns:
    weak_df["Idioma"] = df.loc[
        weak_df.index,
        "Idioma"
    ]

# 8. Control de calidad de las etiquetas


In [49]:
patrones_neg_effusion = [
    r"\bno (joint |knee )?effusion\b",
    r"\bno significant (joint |knee )?effusion\b",
    r"\bsin derrame\b",
    r"\bno hay.{0,40}derrame\b",
    r"няма данни за.{0,40}излив",
    r"без.{0,40}излив"
]

In [50]:
# ============================================================
# QC DE CONSISTENCIA - EFFUSION
# ============================================================

def qc_effusion(reporte, weak_label):

    if weak_label != 1:
        return weak_label

    texto = normalizar_texto(reporte)

    for patron in patrones_neg_effusion:

        if re.search(
            patron,
            texto,
            flags=re.IGNORECASE
        ):
            return -1

    return weak_label


weak_df["Effusion_weak_qc"] = weak_df.apply(
    lambda fila: qc_effusion(
        fila["Report"],
        fila["Effusion_weak"]
    ),
    axis=1
)

In [51]:
print("=== EFFUSION ORIGINAL ===")
print(
    weak_df["Effusion_weak"]
    .value_counts()
    .sort_index()
)

print("\n=== EFFUSION DESPUÉS DEL QC ===")
print(
    weak_df["Effusion_weak_qc"]
    .value_counts()
    .sort_index()
)

cambios_effusion = weak_df[
    weak_df["Effusion_weak"]
    != weak_df["Effusion_weak_qc"]
]

print(
    "\nCasos modificados:",
    len(cambios_effusion)
)

print("\nPor idioma:")
print(
    cambios_effusion["Idioma"]
    .value_counts()
)

=== EFFUSION ORIGINAL ===
Effusion_weak
-1    1810
 0     639
 1    1900
Name: count, dtype: int64

=== EFFUSION DESPUÉS DEL QC ===
Effusion_weak_qc
-1    1968
 0     639
 1    1742
Name: count, dtype: int64

Casos modificados: 158

Por idioma:
Idioma
es    110
bg     48
Name: count, dtype: int64


In [52]:
weak_cols = {
    "ACL": "ACL_weak",
    "MCL": "MCL_weak",
    "Medial Meniscus": "Medial_Meniscus_weak",
    "Lateral Meniscus": "Lateral_Meniscus_weak",
    "Medial OA": "Medial_OA_weak",
    "Lateral OA": "Lateral_OA_weak",
    "PF OA": "PF_OA_weak",

    # Usamos la versión con control de consistencia
    "Effusion": "Effusion_weak_qc",

    "Synovitis": "Synovitis_weak",
    "Baker's": "Bakers_weak",
    "Contusion": "Contusion_weak",
    "Fracture": "Fracture_weak"
}

In [53]:
columnas_weak = list(weak_cols.values())

weak_df["n_resueltas"] = (
    weak_df[columnas_weak] != -1
).sum(axis=1)

weak_df["n_positivas"] = (
    weak_df[columnas_weak] == 1
).sum(axis=1)

# 9. Análisis del conjunto de etiquetas débiles


## Análisis de las etiquetas débiles generadas

In [54]:
# ============================================================
# DISTRIBUCIÓN DE WEAK LABELS
# ============================================================

distribucion_weak = []


for label, col in weak_cols.items():

    positivos = (weak_df[col] == 1).sum()
    negativos = (weak_df[col] == 0).sum()
    abstenciones = (weak_df[col] == -1).sum()

    resueltos = positivos + negativos

    coverage = (
        resueltos / len(weak_df)
        if len(weak_df) > 0
        else 0
    )

    tasa_positivos = (
        positivos / resueltos
        if resueltos > 0
        else 0
    )

    distribucion_weak.append({
        "Label": label,
        "Positivos (1)": positivos,
        "Negativos (0)": negativos,
        "Abstenciones (-1)": abstenciones,
        "Resueltos": resueltos,
        "Coverage (%)": round(coverage * 100, 1),
        "Positivos entre resueltos (%)": round(
            tasa_positivos * 100,
            1
        )
    })


resumen_weak = pd.DataFrame(distribucion_weak)

display(resumen_weak)

,Label,Positivos (1),Negativos (0),Abstenciones (-1),Resueltos,Coverage (%),Positivos entre resueltos (%)
0,ACL,445,1898,2006,2343,53.9,19.0
1,MCL,29,1798,2522,1827,42.0,1.6
2,Medial Meniscus,1223,596,2530,1819,41.8,67.2
3,Lateral Meniscus,700,1304,2345,2004,46.1,34.9
4,Medial OA,283,765,3301,1048,24.1,27.0
5,Lateral OA,255,784,3310,1039,23.9,24.5
6,PF OA,480,730,3139,1210,27.8,39.7
7,Effusion,1742,639,1968,2381,54.7,73.2
8,Synovitis,458,590,3301,1048,24.1,43.7
9,Baker's,795,677,2877,1472,33.8,54.0


In [55]:
print("Total de estudios:", len(weak_df))

for label, col in weak_cols.items():

    total = (
        (weak_df[col] == 1).sum()
        + (weak_df[col] == 0).sum()
        + (weak_df[col] == -1).sum()
    )

    print(
        f"{label:20s} -> {total}"
    )

Total de estudios: 4349
ACL                  -> 4349
MCL                  -> 4349
Medial Meniscus      -> 4349
Lateral Meniscus     -> 4349
Medial OA            -> 4349
Lateral OA           -> 4349
PF OA                -> 4349
Effusion             -> 4349
Synovitis            -> 4349
Baker's              -> 4349
Contusion            -> 4349
Fracture             -> 4349


In [56]:
for n in range(1, 13):

    cantidad = (
        weak_df["n_resueltas"] >= n
    ).sum()

    print(
        f"Al menos {n:2d} etiquetas resueltas: "
        f"{cantidad:4d}"
    )

Al menos  1 etiquetas resueltas: 3943
Al menos  2 etiquetas resueltas: 3396
Al menos  3 etiquetas resueltas: 2760
Al menos  4 etiquetas resueltas: 2264
Al menos  5 etiquetas resueltas: 1814
Al menos  6 etiquetas resueltas: 1433
Al menos  7 etiquetas resueltas: 1095
Al menos  8 etiquetas resueltas:  836
Al menos  9 etiquetas resueltas:  518
Al menos 10 etiquetas resueltas:  260
Al menos 11 etiquetas resueltas:   45
Al menos 12 etiquetas resueltas:   12


In [57]:
# ============================================================
# CALIDAD DE WEAK LABELS POR IDIOMA
# ============================================================

resumen_idiomas = (
    weak_df
    .groupby("Idioma")
    .agg(
        Estudios=("StudyInstanceUID", "count"),
        Prom_resueltas=("n_resueltas", "mean"),
        Prom_positivas=("n_positivas", "mean")
    )
    .reset_index()
)

resumen_idiomas["Con_4_o_mas_resueltas"] = resumen_idiomas["Idioma"].map(
    weak_df.groupby("Idioma")["n_resueltas"]
    .apply(lambda x: (x >= 4).sum())
)

resumen_idiomas["Con_6_o_mas_resueltas"] = resumen_idiomas["Idioma"].map(
    weak_df.groupby("Idioma")["n_resueltas"]
    .apply(lambda x: (x >= 6).sum())
)

resumen_idiomas["Con_8_o_mas_resueltas"] = resumen_idiomas["Idioma"].map(
    weak_df.groupby("Idioma")["n_resueltas"]
    .apply(lambda x: (x >= 8).sum())
)

display(
    resumen_idiomas.sort_values(
        "Estudios",
        ascending=False
    )
)

,Idioma,Estudios,Prom_resueltas,Prom_positivas,Con_4_o_mas_resueltas,Con_6_o_mas_resueltas,Con_8_o_mas_resueltas
3,en,1708,6.628806,2.208431,1507,1120,711
4,es,672,3.889881,1.699405,340,245,125
8,tr,540,2.940741,1.166667,193,38,0
6,hr,402,1.684080,1.191542,33,1,0
2,el,318,2.333333,1.537736,72,14,0
1,de,260,1.619231,1.157692,14,0,0
0,bg,217,3.267281,1.387097,92,15,0
7,nl,151,1.562914,1.211921,13,0,0
5,fr,81,0.827160,0.469136,0,0,0


In [58]:
# ============================================================
# POSITIVOS WEAK POR IDIOMA
# ============================================================

positivos_por_idioma = []

for idioma, grupo in weak_df.groupby("Idioma"):

    fila = {
        "Idioma": idioma,
        "Estudios": len(grupo)
    }

    for label, col in weak_cols.items():
        fila[label] = (grupo[col] == 1).sum()

    positivos_por_idioma.append(fila)

positivos_por_idioma = pd.DataFrame(
    positivos_por_idioma
)

display(positivos_por_idioma)

,Idioma,Estudios,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,bg,217,32,0,39,6,0,0,1,130,1,52,22,18
1,de,260,15,0,14,9,1,0,0,124,2,54,52,30
2,el,318,72,0,123,55,35,0,0,51,28,49,33,43
3,en,1708,176,29,573,352,166,154,330,923,292,235,386,156
4,es,672,16,0,219,108,64,73,128,116,102,225,33,58
5,fr,81,0,0,0,0,0,0,4,0,0,0,0,34
6,hr,402,56,0,71,69,14,19,0,156,4,75,6,9
7,nl,151,0,0,43,17,1,9,0,45,5,48,2,13
8,tr,540,78,0,141,84,2,0,17,197,24,57,13,17


# 10. Selección estratificada de 500 estudios


In [59]:
# ============================================================
# SUBCONJUNTO MULTI-LABEL + DIVERSIDAD LINGÜÍSTICA
# ============================================================

import numpy as np
import pandas as pd

SEED = 42
N_ESTUDIOS = 500
rng = np.random.default_rng(SEED)


# ============================================================
# OBJETIVOS POSITIVOS
# ============================================================

objetivo_positivos = {
    "ACL": 50,
    "MCL": 29,
    "Medial Meniscus": 50,
    "Lateral Meniscus": 50,
    "Medial OA": 50,
    "Lateral OA": 50,
    "PF OA": 50,
    "Effusion": 50,
    "Synovitis": 50,
    "Baker's": 50,
    "Contusion": 50,
    "Fracture": 50
}


# ============================================================
# REPRESENTACIÓN MÍNIMA POR IDIOMA
# ============================================================

min_idiomas = {
    "es": 60,
    "tr": 25,
    "bg": 20,
    "el": 15,
    "hr": 10,
    "de": 10,
    "nl": 10,
    "fr": 5
}


# Solo estudios con al menos una weak label resuelta
candidatos = weak_df[
    weak_df["n_resueltas"] > 0
].copy()

candidatos["_azar"] = rng.random(len(candidatos))


# ============================================================
# 1. TODOS LOS MCL POSITIVOS
# ============================================================

seleccionados = set(
    candidatos.index[
        candidatos["MCL_weak"] == 1
    ]
)

print(
    "MCL positivos incluidos:",
    len(seleccionados)
)

MCL positivos incluidos: 29


In [60]:
# ============================================================
# 2. GARANTIZAR MÍNIMOS POR IDIOMA
# ============================================================

for idioma, minimo in min_idiomas.items():

    actuales = sum(
        candidatos.loc[
            list(seleccionados),
            "Idioma"
        ] == idioma
    )

    faltan = max(
        0,
        minimo - actuales
    )

    if faltan == 0:
        continue

    disponibles = candidatos[
        (candidatos["Idioma"] == idioma) &
        (~candidatos.index.isin(seleccionados))
    ].copy()

    # Preferir los estudios más informativos
    disponibles = disponibles.sort_values(
        by=[
            "n_resueltas",
            "n_positivas",
            "_azar"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )

    elegidos = disponibles.head(
        faltan
    ).index

    seleccionados.update(elegidos)

    print(
        f"{idioma:3s} | "
        f"mínimo={minimo:3d} | "
        f"añadidos={len(elegidos):3d}"
    )

es  | mínimo= 60 | añadidos= 60
tr  | mínimo= 25 | añadidos= 25
bg  | mínimo= 20 | añadidos= 20
el  | mínimo= 15 | añadidos= 15
hr  | mínimo= 10 | añadidos= 10
de  | mínimo= 10 | añadidos= 10
nl  | mínimo= 10 | añadidos= 10
fr  | mínimo=  5 | añadidos=  5


In [61]:
# ============================================================
# 3. GARANTIZAR REPRESENTACIÓN POSITIVA
# ============================================================

for label, col in weak_cols.items():

    objetivo = objetivo_positivos[label]

    actuales = sum(
        weak_df.loc[
            list(seleccionados),
            col
        ] == 1
    )

    faltan = max(
        0,
        objetivo - actuales
    )

    if faltan == 0:
        continue

    disponibles = candidatos[
        (candidatos[col] == 1) &
        (~candidatos.index.isin(seleccionados))
    ].copy()

    disponibles = disponibles.sort_values(
        by=[
            "n_resueltas",
            "n_positivas",
            "_azar"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )

    elegidos = disponibles.head(
        faltan
    ).index

    seleccionados.update(elegidos)

    print(
        f"{label:20s} | "
        f"objetivo={objetivo:3d} | "
        f"añadidos={len(elegidos):3d}"
    )


print(
    "\nDespués de cuotas:",
    len(seleccionados)
)

Medial OA            | objetivo= 50 | añadidos= 24
Lateral OA           | objetivo= 50 | añadidos=  1

Después de cuotas: 209


In [62]:
# ============================================================
# 4. COMPLETAR HASTA 500
# ============================================================

faltan = N_ESTUDIOS - len(seleccionados)

if faltan > 0:

    restantes = candidatos[
        ~candidatos.index.isin(seleccionados)
    ].copy()

    # Priorizar estudios útiles
    restantes["_score"] = (
        restantes["n_resueltas"]
        + 0.25 * restantes["n_positivas"]
        + 0.01 * rng.random(len(restantes))
    )

    restantes = restantes.sort_values(
        "_score",
        ascending=False
    )

    adicionales = restantes.head(
        faltan
    ).index

    seleccionados.update(adicionales)


indices_seleccionados = sorted(
    seleccionados
)


weak_500_v2 = (
    weak_df
    .loc[
        indices_seleccionados
    ]
    .copy()
)


assert len(
    weak_500_v2
) == 500


print(
    "✅ Estudios seleccionados:",
    len(weak_500_v2)
)

✅ Estudios seleccionados: 500


In [63]:
print("=== DISTRIBUCIÓN POR IDIOMA ===")

print(
    weak_500_v2["Idioma"]
    .value_counts()
)

=== DISTRIBUCIÓN POR IDIOMA ===
Idioma
en    345
es     60
tr     25
bg     20
el     15
hr     10
nl     10
de     10
fr      5
Name: count, dtype: int64


In [64]:
print(
    weak_500_v2[
        ["n_resueltas", "n_positivas"]
    ].describe()
)

       n_resueltas  n_positivas
count   500.000000   500.000000
mean      8.814000     3.110000
std       1.979708     2.064564
min       3.000000     0.000000
25%       8.000000     1.000000
50%      10.000000     3.000000
75%      10.000000     5.000000
max      12.000000     9.000000


# 11. Verificación y exportación


In [65]:
print("=== COMPROBACIONES FINALES ===")

print("Número de estudios:", len(weak_500_v2))

print(
    "Número de idiomas:",
    weak_500_v2["Idioma"].nunique()
)

print("\nPositivos por clase:")

for label, col in weak_cols.items():
    positivos = (weak_500_v2[col] == 1).sum()

    print(
        f"{label:20s}: {positivos}"
    )

=== COMPROBACIONES FINALES ===
Número de estudios: 500
Número de idiomas: 9

Positivos por clase:
ACL                 : 99
MCL                 : 29
Medial Meniscus     : 233
Lateral Meniscus    : 135
Medial OA           : 97
Lateral OA          : 96
PF OA               : 165
Effusion            : 241
Synovitis           : 79
Baker's             : 127
Contusion           : 191
Fracture            : 63


In [66]:
# ============================================================
# RESUMEN DEL SUBCONJUNTO FINAL
# ============================================================

resumen_final_500 = []

for label, col in weak_cols.items():

    positivos = (weak_500_v2[col] == 1).sum()
    negativos = (weak_500_v2[col] == 0).sum()
    abstenciones = (weak_500_v2[col] == -1).sum()

    resueltos = positivos + negativos

    resumen_final_500.append({
        "Hallazgo": label,
        "Positivos": positivos,
        "Negativos": negativos,
        "Abstenciones": abstenciones,
        "Coverage (%)": round(
            100 * resueltos / len(weak_500_v2),
            1
        )
    })

resumen_final_500 = pd.DataFrame(resumen_final_500)

display(resumen_final_500)

,Hallazgo,Positivos,Negativos,Abstenciones,Coverage (%)
0,ACL,99,366,35,93.0
1,MCL,29,404,67,86.6
2,Medial Meniscus,233,161,106,78.8
3,Lateral Meniscus,135,269,96,80.8
4,Medial OA,97,175,228,54.4
5,Lateral OA,96,195,209,58.2
6,PF OA,165,220,115,77.0
7,Effusion,241,206,53,89.4
8,Synovitis,79,205,216,56.8
9,Baker's,127,230,143,71.4


In [67]:
ruta_500 = (
    WEAK_DATA_DIR
    / "weak_500_final.csv"
)


weak_500_v2.to_csv(
    ruta_500,
    index=False,
    encoding="utf-8-sig"
)


print(
    "✅ Dataset weak definitivo guardado:"
)

print(
    ruta_500
)

✅ Dataset weak definitivo guardado:
C:\Users\PC\Documents\RSNA_Knee\processed\weak_labels\weak_500_final.csv


# 12. Resumen


## Conclusiones de la generación de etiquetas débiles

Se desarrolló un pipeline basado en reglas multilingües para extraer los
12 hallazgos objetivo a partir de los reportes radiológicos. Los extractores
pueden producir una etiqueta positiva (1), negativa (0) o abstenerse (-1)
cuando el reporte no contiene evidencia textual suficiente.

Durante el desarrollo sobre 40 estudios GOLD se obtuvo un agreement macro
de 92.8%, con un ruido estimado de 7.2% y una cobertura promedio de 61.9%.
Al evaluar los extractores congelados sobre un holdout independiente de
18 estudios GOLD, el agreement macro fue de 73.9%, con un ruido estimado
de 26.1% y una cobertura promedio de 36.6%. Esta diferencia evidencia una
brecha de generalización, especialmente relevante debido al carácter
multilingüe de los reportes.

Posteriormente, los extractores se aplicaron a 4349 estudios sin etiquetas
GOLD. Se realizó además un control de consistencia sobre la etiqueta
Effusion, enviando a abstención los casos con evidencia textual
contradictoria en lugar de forzar una etiqueta negativa.

Finalmente, se construyó un subconjunto estratificado de 500 estudios,
priorizando la representación de casos positivos para los 12 hallazgos y
la diversidad lingüística. Este subconjunto será utilizado como conjunto
de entrenamiento con etiquetas débiles para los modelos de clasificación
multietiqueta de imágenes MRI.

Las etiquetas -1 se conservarán como abstenciones y serán ignoradas mediante
una máscara durante el cálculo de la función de pérdida, evitando convertir
la ausencia de evidencia textual en una etiqueta negativa artificial.